# TabTune · Real-World Cash-Flow Forecasting
### The US Daily Treasury Statement: 490,000 rows of daily government cash flows

**Library:** TabTune open source (`tabtune.TimeSeries`) · **Runtime:** Colab, A100 GPU, High-RAM

**The data.** Every business day the US Treasury publishes the deposits into and withdrawals from its operating cash account, line by line: tax receipts, Social Security, Medicare, defence vendors, interest, tax refunds and about 150 other categories. It is public, updated daily, and goes back to 2005. It is the closest public analogue to a bank treasury's cash ledger: large, daily, calendar-driven, and messy.

**Why it is hard.** The raw feed was never designed for modelling:
* categories are **free-text labels that were renamed four times** (2012, 2019–22, 2023) — the same flow appears under different names in different years;
* totals, sub-totals, debt financing and internal transfers sit in the same table as operating flows;
* amounts can be negative (reversals), and there are no rows on federal holidays;
* the report layout itself changed (three different account structures).

## Notebook map

| # | Section | What you get |
|---|---|---|
| 1 | Onboarding | Install, model options in TabTune time series and on the Lexsi platform |
| 2 | Product options | Problem statement, training and test data, development vs production choices |
| 3 | Data preparation | 490,000 raw Treasury rows: structural rows, label harmonisation across four reporting eras, business-day grid; a **data readiness agent** with approval gates; feature engineering dashboards |
| 4 | Model training | AutoML race on rolling backtests, hyperparameter sweeps, selection rule, covariate explainability, ensembles |
| 5 | Model testing | Fan charts, hold-out dashboard, net operating cash dashboard |
| 6 | Batch inference | Every series forecast in one call, written to a file |
| 7 | Model risk | Calibrated bands with measured coverage, accuracy guards, regulatory evidence map |
| 8 | Artefacts before production | Model card written by an open-source LLM with trace and grounding check, forecast API contract |
| 9 | Model in production | Anomaly replay of the COVID fiscal shock, drift monitoring, **retraining agent** with human approval, regenerated model card |
| 10 | Conclusions | Plain-English summary and the audit trail |

Source: US Department of the Treasury, Bureau of the Fiscal Service, [FiscalData — Daily Treasury Statement](https://fiscaldata.treasury.gov/datasets/daily-treasury-statement/) (public domain). Amounts are in USD millions.

**Run:** *Runtime → Change runtime type → A100 GPU, High-RAM*, then *Run all*. Set `AUTO_APPROVE = False` in the imports cell to be asked at each approval gate. The download takes about two minutes and is cached.

# 1 · Onboarding

In [ ]:
# ⚙️ Install TabTune (open source) and the Lexsi SDK — about 3 minutes on Colab
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "tabtune[colab] @ git+https://github.com/Lexsi-Labs/TabTune.git",
                    "lexsi-sdk", "plotly", "ipywidgets", "accelerate"], check=True)
from importlib.metadata import version, PackageNotFoundError
for pkg in ("tabtune", "lexsi-sdk", "torch"):
    try:
        print(f"{pkg:10} {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg:10} not installed (only needed for the platform reference cells)" if pkg == "lexsi-sdk" else f"{pkg:10} MISSING")

In [ ]:
# 🔧 Imports, theme and the run-wide fact sheet
import json, logging, re, time, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown
import ipywidgets as W

warnings.filterwarnings("ignore")
from tabtune.logger import setup_logger
setup_logger(level=logging.WARNING)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.max_columns", 60)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none"
except ImportError:
    DEVICE, GPU = "cpu", "none"

FAST = os.environ.get("TABTUNE_DEMO_FAST") == "1"      # tiny settings for a smoke test without a GPU
AUTO_APPROVE = True                                    # False = the agents stop and ask you to type yes / no
BLUE, ORANGE, TEAL, GREY, DARK, RED, GOLD = "#2F5BEA", "#B4541A", "#1B998B", "#9AA0A8", "#16181D", "#D7263D", "#E8C547"
PALETTE = [BLUE, ORANGE, TEAL, RED, GOLD, GREY, DARK, "#7A5CFA"]
FACTS = {}                                             # every number that reaches the model card is recorded here
AUDIT = []                                             # every agent step and human decision


def record(section, **values):
    """Store results for the model card and the conclusions."""
    FACTS.setdefault(section, {}).update({k: (round(float(v), 4) if isinstance(v, (int, float, np.floating, np.integer)) and not isinstance(v, bool) else v)
                                          for k, v in values.items()})


def audit(actor, step, finding, decision=""):
    AUDIT.append({"at": pd.Timestamp.now().strftime("%H:%M:%S"), "actor": actor, "step": step, "finding": finding, "decision": decision})
    print(f"[{actor} · {step}] {finding}" + (f"  →  {decision}" if decision else ""))


def human_gate(question, recommended=True):
    """Human-in-the-loop checkpoint. With AUTO_APPROVE the recommendation is taken and logged; otherwise you are asked."""
    if AUTO_APPROVE:
        answer = recommended
        who = "auto-approved (AUTO_APPROVE=True)"
    else:
        reply = input(f"🧑 {question}  [recommended: {'yes' if recommended else 'no'}]  yes/no: ").strip().lower()
        answer = reply.startswith("y") if reply else recommended
        who = "human reviewer"
    audit("human", "approval gate", question, f"{'APPROVED' if answer else 'REJECTED'} · {who}")
    return answer


from tabtune.TimeSeries import (TimeSeriesPipeline, TimeSeriesSchema, TimeSeriesLeaderboard,
                                TimeSeriesEnsemble, split_horizon)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

# FAST=1 swaps foundation models for weight-free models (smoke test without GPU or downloads)
QUANTILES = [0.05, 0.1, 0.5, 0.9, 0.95]
FOUNDATION_MODELS = [] if FAST else ["ChronosBolt", "Chronos2"]     # Apache-2.0 weights: commercial-safe
BASELINES = ["SeasonalNaive", "WindowAverage", "TabularTS-GBM"]
PRIMARY = "TabularTS-GBM" if FAST else "Chronos2"                   # reads covariates
DETECTOR = "SeasonalNaive" if FAST else "ChronosBolt"
EMBEDDER = None if FAST else "ChronosBolt"
RUN_FINETUNE = False                                                # True adds a LoRA fine-tuned challenger (GPU, a few minutes)
GBM_PARAMS = {"max_iter": 60}                                       # keeps the tree baseline quick
PRIMARY_PARAMS = GBM_PARAMS if PRIMARY == "TabularTS-GBM" else {}
DOW = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
MON = ["", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
print(f"device: {DEVICE} ({GPU}) · primary model: {PRIMARY} · detector: {DETECTOR} · fast mode: {FAST} · auto-approve: {AUTO_APPROVE}")

### Model support in TabTune time series

The dictionary below is built from the library's own registry, so it always matches the installed version.

In [ ]:
# 📚 Time-series models available in TabTune today (read live from the library's registry)
from tabtune.registry import list_time_series_models

def model_kind(spec):
    if spec.name.startswith("TabularTS") or spec.name == "TabPFN-TS":
        return "Tabular model as forecaster"
    return "Foundation model" if spec.default_checkpoint else "Statistical baseline"

TS_MODELS = {
    spec.name: {
        "kind": model_kind(spec),
        "family": spec.family,
        "tasks": sorted(spec.tasks),
        "strategies": sorted(spec.strategies),          # inference = zero-shot, finetune, peft = LoRA
        "default_checkpoint": spec.default_checkpoint or "none (no weights)",
        "max_context": spec.max_context,
        "multivariate": spec.supports_multivariate,
        "covariates": spec.supports_covariates,
        "licence": spec.license.name,
        "commercial_use": {True: "yes", False: "no", None: "unverified"}[spec.license.commercial_use_ok],
    }
    for spec in list_time_series_models()
}

catalogue_ts = pd.DataFrame(TS_MODELS).T.rename_axis("model")
print(f"{len(TS_MODELS)} models registered:", catalogue_ts["kind"].value_counts().to_dict())
for kind in ("Foundation model", "Tabular model as forecaster", "Statistical baseline"):
    part = catalogue_ts[catalogue_ts.kind == kind].drop(columns="kind")
    part = part.assign(tasks=part.tasks.map(", ".join), strategies=part.strategies.map(", ".join))
    display(part.style.set_caption(f"{kind} · {len(part)} models"))
TS_MODELS["Chronos2"]          # one entry of the dictionary

### Options on the Lexsi platform
Time-series forecasting and anomaly detection run in the **TabTune library** today, which is what this notebook uses. The Lexsi platform is where classification and regression models are trained, registered and served; its options are listed here so the two can be read side by side.

In [ ]:
# 🗂️ The same workflow on the Lexsi platform (reference: nothing in this cell calls the platform)
LEXSI_OPTIONS = {
    "structure": "Organization → Workspace → Project (one project per use case; project type is classification or regression)",
    "data": "project.upload_data(file, tag): tags such as Training / Test / Batch1 select data for training, testing and inference",
    "training_strategies": ["simple", "ensemble", "distillation", "leaderboard"],
    "tuning_strategies": ["inference (zero-shot)", "finetune", "peft (LoRA)"],
    "compute": {"CPU": ["xsmall", "small", "medium", "large", "xlarge", "2xlarge", "3xlarge"],
                "GPU": ["T4.xsmall", "T4.small", "A10G.xmedium"]},
    "foundation_models": ["TabPFN", "TabPFNv26", "TabPFNv3", "TabICL", "TabICLv2", "OrionMSP", "OrionMSPv1.5", "OrionBix", "OrionMoE",
                          "Mitra", "ContextTab", "TabDPT", "Limix", "TabFM", "EXAONETabular", "ILTM", "XRFM"],
    "classic_models": ["XGBoost", "LGBoost", "CatBoost", "RandomForest", "SGD", "LogisticRegression", "LinearRegression", "GaussianNaiveBayes"],
    "explainability": ["shap", "lime", "support_set (training-row attribution, in-context models)"],
    "inference": "project.model_inference(tag=..., model_name=..., compute_type=...) for batches; project.case_predict(...) for one case",
    "governance": "model registry per project, activate_model for promotion, case audit trail, policies and observations",
}
display(pd.DataFrame({"On the Lexsi platform": {k: (", ".join(v) if isinstance(v, list) else json.dumps(v) if isinstance(v, dict) else v)
                                               for k, v in LEXSI_OPTIONS.items()}}).style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))
display(Markdown("""
**Platform equivalent of this notebook's training step** (shown for reference, not executed here):
```python
from lexsi_sdk import lexsi
lexsi.login()
project = lexsi.organization("…").workspace("…").project("…")
project.upload_data("train.csv", "Training", config={"unique_identifier": "id", "true_label": "TARGET"}, compute_type="small")
project.train_model(model="OrionMSP", training_strategy="simple", tunning_strategy="inference", compute_type="T4.small",
                    data_config={"tags": ["Training"], "test_tags": ["Test"], "xai_method": ["shap", "lime"]})
```"""))

# 2 · Product options

## Problem statement
**What will the net operating cash position be over the next 20 business days, line by line?** A treasury forecasts 18 daily cash-flow lines (tax receipts, benefits, interest, refunds and more) to plan funding, and wants to know whether a monitor would have caught a regime break such as March 2020.

## Training and test data

| Set | What it is | Used for |
|---|---|---|
| History (training / context) | all business days from October 2016 up to the final 20 | context for foundation models; training data for the tree forecaster |
| Rolling backtests | the last three 20-day windows inside the history | model race and selection |
| Inner window | one 20-day window inside the history | hyperparameter sweeps and covariate explainability |
| Hold-out (test) | the final 20 business days | untouched final report |
| Replay | January 2019 to June 2021 | anomaly monitoring around the COVID fiscal shock |

## Development vs production

| Choice | In development (this notebook) | In production |
|---|---|---|
| Engine | TabTune open source (`tabtune.TimeSeries`) on the notebook GPU | the same library on Lexsi-managed compute |
| Models | 33 registered forecasters: 9 foundation models, 19 tabular-as-forecaster, 5 baselines | the commercially licensed subset, fixed per use case |
| Adaptation | zero-shot, optional LoRA fine-tuning | zero-shot by default; fine-tuning as a managed job |
| Scoring | batch forecast in the notebook | scheduled batch, plus the forecast API in section 8 |
| Governance | audit list and model card in this notebook | registry, approvals, audit trail and policies on the Lexsi platform |

In [ ]:
# 🔧 Use-case settings
USE_CASE_NAME, DECISION_SUPPORTED = 'Treasury cash-flow forecasting (US Daily Treasury Statement)', 'funding plan from the 20-business-day net cash outlook; regime-break alerts'
CLOSING_LINE = "**Reading it honestly:** this is a public government ledger used as a stand-in for a bank treasury's cash ledger; the anomaly replay has no ground-truth labels, so it shows alert timing, not a detection rate."
HORIZON, SEASON, MONTH_LAG, SCALE_WINDOW = 20, 5, 21, 250      # business days: 4-week horizon, 5-day week
UNIT = "USD mn"
MODEL_START = "2016-10-03"                                      # start of fiscal year 2017
GROUP_COLORS = {"Inflow": TEAL, "Outflow": ORANGE}
DTS_CACHE = os.environ.get("DTS_CACHE", "/content/dts_raw.parquet")

# 3 · Data preparation

## 3.1 Download and first look

In [ ]:
# 📥 Daily Treasury Statement, table II (deposits and withdrawals of operating cash), full history
import requests

API = "https://api.fiscaldata.treasury.gov/services/api/fiscal_service/v1/accounting/dts/deposits_withdrawals_operating_cash"
FIELDS = "record_date,account_type,transaction_type,transaction_catg,transaction_catg_desc,transaction_today_amt"

if os.path.exists(DTS_CACHE):
    raw = pd.read_parquet(DTS_CACHE)
    print("loaded cache:", DTS_CACHE)
else:
    rows, page, t0 = [], 1, time.time()
    while True:
        r = requests.get(API, params={"fields": FIELDS, "sort": "record_date", "page[size]": 10000, "page[number]": page}, timeout=120)
        r.raise_for_status()
        j = r.json()
        rows += j["data"]
        if page >= j["meta"]["total-pages"]:
            break
        page += 1
    raw = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(DTS_CACHE) or ".", exist_ok=True)
    raw.to_parquet(DTS_CACHE)
    print(f"downloaded {page} pages in {time.time() - t0:.0f}s")

raw["date"] = pd.to_datetime(raw["record_date"])
raw["amount"] = pd.to_numeric(raw["transaction_today_amt"], errors="coerce")
raw["category"] = raw["transaction_catg"].str.strip()
print(f"{len(raw):,} rows · {raw.date.nunique():,} statement days · {raw.date.min():%d %b %Y} → {raw.date.max():%d %b %Y} · "
      f"{raw.category.nunique()} distinct category labels")
display(raw[["record_date", "account_type", "transaction_type", "transaction_catg", "transaction_today_amt"]].sample(6, random_state=1))

In [ ]:
# 🔎 Raw profile: the feed changes shape over time
per_day = raw.groupby("date").size()
labels_per_year = raw.groupby(raw.date.dt.year)["category"].nunique()
acct = raw.groupby("account_type")["date"].agg(["min", "max"]).sort_values("min")
neg = raw[raw.amount < 0].groupby(raw.date.dt.year).size().reindex(labels_per_year.index, fill_value=0)

fig = make_subplots(rows=2, cols=2, vertical_spacing=0.16,
                    subplot_titles=("Rows per statement day", "Distinct category labels in use, by year",
                                    "Account structure over time", "Negative amounts (reversals) per year"))
fig.add_trace(go.Scattergl(x=per_day.index, y=per_day.values, mode="markers", marker=dict(size=3, color=BLUE), showlegend=False), 1, 1)
fig.add_trace(go.Bar(x=labels_per_year.index, y=labels_per_year.values, marker_color=ORANGE, showlegend=False), 1, 2)
for i, (name, r) in enumerate(acct.iterrows()):
    fig.add_trace(go.Scatter(x=[r["min"], r["max"]], y=[name[:34], name[:34]], mode="lines", line=dict(width=12, color=[BLUE, ORANGE, TEAL, GREY, RED, GOLD, DARK][i % 7]),
                             showlegend=False), 2, 1)
fig.add_trace(go.Bar(x=neg.index, y=neg.values, marker_color=RED, showlegend=False), 2, 2)
fig.update_layout(template="plotly_white", height=720, title="The raw feed · three layouts, shifting label sets, reversals")
fig.show()

## 3.2 Cleaning the feed, visualised

### 3.2a · Separate operating flows from structural rows
The table mixes four kinds of row. Only **operating** flows are a forecasting target; debt issuance and redemption are financing decisions, transfers move cash between the Treasury's own accounts, and totals would double count.

In [ ]:
# 🧱 Classify every row: operating, financing, internal transfer, or total
def row_kind(cat, acct):
    c = cat.lower()
    if c in ("null", "") or "total" in c or "total" in acct.lower():
        return "total / sub-total"
    if c.startswith("public debt"):
        return "debt financing"
    if c.startswith("transfers ") or "(table v)" in c:
        return "internal transfer"
    return "operating"

kinds = {(c, a): row_kind(c, a) for c, a in raw[["category", "account_type"]].drop_duplicates().itertuples(index=False)}
raw["kind"] = [kinds[k] for k in zip(raw.category, raw.account_type)]
by_kind = raw.groupby([raw.date.dt.year, "kind"])["amount"].sum().unstack(fill_value=0) / 1e6
KIND_COLORS = {"operating": BLUE, "debt financing": GREY, "internal transfer": GOLD, "total / sub-total": RED}
fig = make_subplots(rows=1, cols=2, column_widths=[0.65, 0.35], subplot_titles=("Gross amount by row type and year (USD trillion)", "Rows by type"))
for k in KIND_COLORS:
    fig.add_trace(go.Bar(x=by_kind.index, y=by_kind[k], name=k, marker_color=KIND_COLORS[k]), 1, 1)
cnt = raw.kind.value_counts()
fig.add_trace(go.Bar(x=cnt.index, y=cnt.values, marker_color=[KIND_COLORS[k] for k in cnt.index], text=cnt.values,
                     texttemplate="%{text:,}", showlegend=False), 1, 2)
fig.update_layout(template="plotly_white", height=460, barmode="stack", legend=dict(orientation="h", y=-0.15),
                  title="Most of the money in the table is not an operating flow")
fig.show()
ops = raw[raw.kind == "operating"].copy()
print(f"operating rows kept: {len(ops):,} of {len(raw):,}")

### 3.2b · Harmonise category labels across eras
The same economic flow carries different labels in different years (for example `Social Security Benefits (EFT)` became `SSA - Benefits Payments` in 2021, and Medicare went from one line to two to three). We map the raw labels onto **16 canonical cash-flow lines** with explicit rules, and collect everything else into two residual lines so the net position stays complete.

In [ ]:
# 🏷️ Canonical lines: (direction, label patterns). Patterns are regular expressions on the raw label.
LINES = {
    "Federal tax deposits":           ("Deposits",    [r"^(cash )?ftd's received", r"^taxes - (withheld|corporate|miscellaneous excise|federal unemployment|railroad)"]),
    "Individual taxes, not withheld": ("Deposits",    [r"^individual income and employment taxes, not withheld", r"^taxes - non withheld"]),
    "Customs and excise":             ("Deposits",    [r"customs"]),
    "Postal Service receipts":        ("Deposits",    [r"^postal service$", r"^united states postal service"]),
    "Education receipts":             ("Deposits",    [r"^education department programs", r"^dept of education"]),
    "Social Security benefits":       ("Withdrawals", [r"^social security benefits \(eft\)", r"^ssa - benefits payments"]),
    "Medicare":                       ("Withdrawals", [r"^medicare$", r"^medicare and other cms", r"^medicare advantage", r"^hhs - federal (supple med|hospital) insr", r"^hhs - medicare prescription"]),
    "Medicaid":                       ("Withdrawals", [r"^medicaid$", r"^hhs - grants to states for medicaid"]),
    "Defense payments":               ("Withdrawals", [r"^defense vendor payments", r"^dept of defense \(dod\) - misc"]),
    "Federal salaries":               ("Withdrawals", [r"^federal salaries"]),
    "Interest on Treasury securities": ("Withdrawals", [r"^interest on treasury securities"]),
    "Education payments":             ("Withdrawals", [r"^education department programs", r"^dept of education"]),
    "Unemployment benefits":          ("Withdrawals", [r"^unemployment insurance benefits", r"^dol - unemployment benefits"]),
    "SNAP (food assistance)":         ("Withdrawals", [r"nutrition assist"]),
    "Veterans programs":              ("Withdrawals", [r"^veterans affairs programs", r"^dept of veterans affairs", r"^va - benefits"]),
    "Individual tax refunds":         ("Withdrawals", [r"^irs tax refunds individual", r"^taxes - individual tax refunds"]),
}

def canonical(cat, direction):
    c = cat.lower()
    for line, (d, patterns) in LINES.items():
        if d == direction and any(re.search(p, c) for p in patterns):
            return line
    return "Other deposits" if direction == "Deposits" else "Other withdrawals"

pairs = ops[["category", "transaction_type"]].drop_duplicates()
lookup = {(c, t): canonical(c, t) for c, t in pairs.itertuples(index=False)}
ops["line"] = [lookup[k] for k in zip(ops.category, ops.transaction_type)]

mapped = (ops[~ops.line.str.startswith("Other")].groupby(["line", "category"])
          .agg(first=("date", "min"), last=("date", "max"), total=("amount", "sum")).reset_index())
mapped = mapped[mapped["last"] >= "2012-01-01"].sort_values(["line", "first"])
fig = px.timeline(mapped, x_start="first", x_end="last", y="category", color="line", height=30 * len(mapped) + 160,
                  title=f"{len(mapped)} raw labels stitched into {mapped.line.nunique()} canonical lines · each bar is one label's lifetime")
fig.update_yaxes(categoryorder="array", categoryarray=mapped.category.tolist()[::-1], title="")
fig.update_layout(template="plotly_white", legend=dict(orientation="h", y=-0.06, title=""))
fig.show()

share = ops[ops.date >= "2012-01-01"].assign(named=lambda d: ~d.line.str.startswith("Other")).groupby([ops.date.dt.year, "named"])["amount"].sum().unstack()
print("share of operating volume covered by the 16 named lines, by year (%):")
display((share[True] / share.sum(axis=1) * 100).round(1).to_frame("named lines %").T)

In [ ]:
# ✅ Continuity check: does a line keep its level across each relabelling?
daily = ops.groupby(["line", "date"])["amount"].sum().rename("value").reset_index()
breaks = []
for line, g in mapped.groupby("line"):
    for d0 in sorted(g["first"].unique())[1:]:
        s = daily[daily.line == line].set_index("date")["value"]
        before, after = s[d0 - pd.Timedelta(days=365):d0 - pd.Timedelta(days=1)].sum(), s[d0:d0 + pd.Timedelta(days=364)].sum()
        breaks.append({"line": line, "relabelled on": d0.date(), "12 months before": before, "12 months after": after,
                       "change_%": (after / before - 1) * 100 if before else np.nan})
breaks = pd.DataFrame(breaks)
display(breaks.style.format({"12 months before": "{:,.0f}", "12 months after": "{:,.0f}", "change_%": "{:+.1f}%"})
        .background_gradient(subset=["change_%"], cmap="RdBu_r", vmin=-40, vmax=40)
        .set_caption("Annual volume either side of each relabelling (USD mn). Small changes mean the stitch is clean; "
                     "large ones are real events (for example COVID in 2020) or a scope change worth a note."))

show = "Medicare"
era = ops[ops.line == show].groupby(["category", pd.Grouper(key="date", freq="MS")])["amount"].sum().reset_index()
stitched = daily[daily.line == show].set_index("date")["value"].resample("MS").sum()
fig = go.Figure()
for cat, g in era.groupby("category"):
    fig.add_trace(go.Bar(x=g.date, y=g.amount, name=cat))
fig.add_trace(go.Scatter(x=stitched.index, y=stitched.values, name="stitched line", line=dict(color=DARK, width=2)))
fig.update_layout(template="plotly_white", height=460, barmode="stack", legend=dict(orientation="h", y=-0.2),
                  title=f"{show} · six raw labels over three eras become one continuous monthly series (USD mn)")
fig.show()

### 3.2c · Regular business-day grid, holidays and reversals
Models need one value per series per step. There is no statement on federal holidays, so those days are added with a zero flow and flagged. Reversals (negative amounts) are kept: they are real cash movements.

In [ ]:
# 📅 Business-day grid, holiday handling, final panel
from pandas.tseries.holiday import USFederalHolidayCalendar

daily = daily[daily.date >= MODEL_START]
bdays = pd.bdate_range(daily.date.min(), daily.date.max())
missing = bdays.difference(pd.DatetimeIndex(daily.date.unique()))
fed = USFederalHolidayCalendar().holidays(bdays.min(), bdays.max())
print(f"{len(bdays):,} business days · {len(missing)} without a statement · {missing.isin(fed).sum()} of those are federal holidays")
other = missing[~missing.isin(fed)]
if len(other):
    print("non-holiday gaps (one-off closures):", [d.strftime("%Y-%m-%d") for d in other[:12]])

lines = sorted(daily.line.unique())
grid = pd.MultiIndex.from_product([lines, bdays], names=["item_id", "timestamp"])
panel = (daily.rename(columns={"line": "item_id", "date": "timestamp"}).set_index(["item_id", "timestamp"])
         .reindex(grid, fill_value=0.0).reset_index())
direction = {**{k: v[0] for k, v in LINES.items()}, "Other deposits": "Deposits", "Other withdrawals": "Withdrawals"}
meta = pd.DataFrame({"item_id": lines})
meta["group"] = meta.item_id.map(lambda l: "Inflow" if direction[l] == "Deposits" else "Outflow")
meta = meta.sort_values(["group", "item_id"]).reset_index(drop=True)

yr = panel.assign(group=panel.item_id.map(meta.set_index("item_id")["group"]))
tot = yr.groupby("item_id")["value"].sum().sort_values()
neg_days = (panel.value < 0).groupby(panel.item_id).sum().reindex(tot.index)
zero_days = (panel.value == 0).groupby(panel.item_id).mean().mul(100).reindex(tot.index)
fig = make_subplots(rows=1, cols=3, shared_yaxes=True, subplot_titles=("Total since FY2017 (USD trillion)", "Days with zero flow (%)", "Days with a net reversal"))
fig.add_trace(go.Bar(y=tot.index, x=tot.values / 1e6, orientation="h", showlegend=False,
                     marker_color=[GROUP_COLORS[meta.set_index("item_id").group[i]] for i in tot.index]), 1, 1)
fig.add_trace(go.Bar(y=tot.index, x=zero_days.values, orientation="h", marker_color=GREY, showlegend=False), 1, 2)
fig.add_trace(go.Bar(y=tot.index, x=neg_days.values, orientation="h", marker_color=RED, showlegend=False), 1, 3)
fig.update_layout(template="plotly_white", height=620, title="The modelling panel · teal = inflow, orange = outflow")
fig.show()
print(f"panel: {len(panel):,} rows · {panel.item_id.nunique()} series · {panel.timestamp.min():%d %b %Y} → {panel.timestamp.max():%d %b %Y}")

## 3.3 Feature engineering, visualised

Government cash follows rules written in law: Social Security pays on the 3rd and on the second, third and fourth Wednesdays; estimated taxes fall due on fixed dates; salaries follow a two-week cycle; interest lands mid-month and month-end. We turn those rules into **calendar covariates known in advance**.

In [ ]:
# 🗓️ Payment-calendar and tax-calendar features (all known in advance)
def next_bday(d):
    d = pd.Timestamp(d)
    while d.dayofweek >= 5 or d in fed:
        d += pd.Timedelta(days=1)
    return d

ts = pd.DatetimeIndex(bdays)
cal = pd.DataFrame({"timestamp": ts})
cal["day_of_week"] = ts.dayofweek
cal["day_of_month"] = ts.day
month_key = ts.to_period("M")
cal["bday_of_month"] = cal.groupby(month_key).cumcount().to_numpy() + 1
cal["bdays_to_month_end"] = cal.groupby(month_key).cumcount(ascending=False).to_numpy()
cal["is_month_start"] = (cal.bday_of_month == 1).astype(int)
cal["is_month_end"] = (cal.bdays_to_month_end == 0).astype(int)
years = range(ts.year.min(), ts.year.max() + 2)
mids = {next_bday(f"{y}-{m:02d}-15") for y in years for m in range(1, 13)}
thirds = {next_bday(f"{y}-{m:02d}-03") for y in years for m in range(1, 13)}
tax_due = {next_bday(f"{y}-{md_}") for y in years for md_ in ("01-15", "03-15", "04-15", "06-15", "09-15", "12-15")}
cal["is_mid_month"] = ts.isin(mids).astype(int)
cal["is_ss_third"] = ts.isin(thirds).astype(int)                                        # Social Security: the 3rd
wed_no = (ts.day - 1) // 7 + 1
cal["is_ss_wednesday"] = ((ts.dayofweek == 2) & np.isin(wed_no, [2, 3, 4])).astype(int)  # and 2nd–4th Wednesdays
cal["is_tax_due_date"] = ts.isin(tax_due).astype(int)
cal["is_tax_due_week"] = pd.Series(ts.isin(tax_due)).rolling(5, min_periods=1).max().astype(int).to_numpy()
cal["is_refund_season"] = ts.month.isin([2, 3, 4, 5]).astype(int)
cal["is_holiday"] = ts.isin(fed).astype(int)
cal["is_post_holiday"] = cal.is_holiday.shift(1, fill_value=0)
cal["is_fiscal_year_end"] = ((ts.month == 9) & (cal.bdays_to_month_end <= 4)).astype(int)

panel = panel.merge(cal, on="timestamp")
KNOWN = [c for c in cal.columns if c != "timestamp"]
PAST = []
COV_DOC = {"day_of_week": "0 = Monday … 4 = Friday", "day_of_month": "calendar day", "bday_of_month": "business day number within the month",
           "bdays_to_month_end": "business days left in the month", "is_month_start": "first business day of the month",
           "is_month_end": "last business day of the month", "is_mid_month": "the 15th, or the next business day",
           "is_ss_third": "Social Security payment date (the 3rd, or the next business day)",
           "is_ss_wednesday": "Social Security payment Wednesdays (2nd, 3rd, 4th)",
           "is_tax_due_date": "estimated and corporate tax due dates (15 Jan, Mar, Apr, Jun, Sep, Dec)",
           "is_tax_due_week": "the due date and the four business days after it",
           "is_refund_season": "February to May", "is_holiday": "federal holiday (no statement)",
           "is_post_holiday": "first business day after a holiday", "is_fiscal_year_end": "last five business days of September"}
DECOMPOSE_ITEM = "Social Security benefits"

d = panel[panel.timestamp >= panel.timestamp.max() - pd.Timedelta(days=75)]
focus = ["Social Security benefits", "Federal tax deposits", "Interest on Treasury securities"]
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.04, row_heights=[0.26, 0.26, 0.26, 0.22],
                    subplot_titles=focus + ["Calendar flags"])
for i, item in enumerate(focus, 1):
    g = d[d.item_id == item]
    fig.add_trace(go.Bar(x=g.timestamp, y=g.value, marker_color=[BLUE, TEAL, ORANGE][i - 1], showlegend=False), i, 1)
flag_rows = ["is_ss_third", "is_ss_wednesday", "is_tax_due_date", "is_mid_month", "is_month_end", "is_holiday"]
z = d[d.item_id == focus[0]].set_index("timestamp")[flag_rows].T
fig.add_trace(go.Heatmap(z=z.values, x=z.columns, y=z.index, colorscale=[[0, "#F7F7F4"], [1, DARK]], showscale=False), 4, 1)
fig.update_layout(template="plotly_white", height=820, title="The last 75 days: flows line up with the engineered calendar flags (USD mn)")
fig.show()

### Data readiness agent
Before modelling, a small rule-based agent checks every series in the prepared panel, **proposes** an action per finding, waits at a **human approval gate**, applies what was approved and logs it.

In [ ]:
# 🤖 Data readiness agent: inspect → propose → approve → apply → log
def series_checks(g):
    v = g["value"].astype(float); step = g["timestamp"].diff().dropna()
    med = v.median(); mad = (v - med).abs().median() * 1.4826 + 1e-9
    return pd.Series({"observations": len(g), "largest_gap_steps": float(step.max() / step.median()) if len(step) else 0.0,
                      "missing_%": v.isna().mean() * 100, "zero_%": (v == 0).mean() * 100, "negative_%": (v < 0).mean() * 100,
                      "extreme_days": int((((v - med).abs() / mad) > 12).sum())})

readiness = panel.sort_values("timestamp").groupby("item_id").apply(series_checks)
cov_cols = [c for c in KNOWN + PAST if c in panel]
cov_missing = panel[cov_cols].isna().mean().mul(100) if cov_cols else pd.Series(dtype=float)
MIN_OBS = 2 * SCALE_WINDOW

short = readiness.index[readiness.observations < MIN_OBS].tolist()
audit("readiness-agent", "history length", f"{len(short)} series have fewer than {MIN_OBS} observations", "propose: exclude" if short else "ok")
if short and human_gate(f"Exclude {len(short)} short series from modelling?", True):
    panel = panel[~panel.item_id.isin(short)].reset_index(drop=True); meta = meta[~meta.item_id.isin(short)].reset_index(drop=True)
gappy = readiness.index[readiness["largest_gap_steps"] > 6].tolist()
audit("readiness-agent", "calendar gaps", f"{len(gappy)} series have a gap longer than 6 steps", "flag for the data owner" if gappy else "ok")
holes = readiness.index[readiness["missing_%"] > 0].tolist()
audit("readiness-agent", "missing target values", f"{len(holes)} series contain missing values", "propose: interpolate" if holes else "ok")
if holes and human_gate(f"Interpolate missing values in {len(holes)} series?", True):
    panel["value"] = panel.groupby("item_id")["value"].transform(lambda s: s.interpolate(limit_direction="both"))
bad_cov = cov_missing[cov_missing > 0].index.tolist()
audit("readiness-agent", "covariate completeness", f"{len(bad_cov)} of {len(cov_cols)} covariates have missing values", "propose: forward-fill" if bad_cov else "ok")
if bad_cov and human_gate(f"Forward-fill {len(bad_cov)} covariates?", True):
    panel[bad_cov] = panel.groupby("item_id")[bad_cov].ffill().bfill()
n_ext = int(readiness.extreme_days.sum())
audit("readiness-agent", "extreme days", f"{n_ext} series-days sit more than 12 robust deviations from their series median",
      "propose: keep (real events; the monitor in section 9 handles them)")
human_gate(f"Keep the {n_ext} extreme days in the training history?", True)

show = readiness[["missing_%", "zero_%", "negative_%", "extreme_days", "largest_gap_steps"]]
norm_ = (show.assign(largest_gap_steps=show.largest_gap_steps - 1) / show.max().replace(0, 1)).clip(0, 1).fillna(0)
fig = make_subplots(rows=1, cols=2, column_widths=[0.68, 0.32], subplot_titles=("Readiness checks by series (colour = share of the worst series)", "History length (observations)"))
fig.add_trace(go.Heatmap(z=norm_.values, x=[c.replace("_", " ") for c in show.columns], y=show.index, colorscale=[[0, "#F7F7F4"], [1, ORANGE]],
                         text=show.round(1).values, texttemplate="%{text}", showscale=False), 1, 1)
fig.add_trace(go.Bar(x=readiness.observations, y=readiness.index, orientation="h", marker_color=TEAL, showlegend=False), 1, 2)
fig.update_yaxes(showticklabels=False, row=1, col=2)
fig.update_layout(template="plotly_white", height=max(480, 22 * len(readiness)), title=f"Data readiness agent · {len(readiness)} series checked · {panel.item_id.nunique()} go forward")
fig.show()
record("data", series=int(panel.item_id.nunique()), panel_rows=len(panel), first_date=str(panel.timestamp.min().date()), last_date=str(panel.timestamp.max().date()),
       known_covariates=len(KNOWN), past_covariates=len(PAST), unit=UNIT)
record("preparation", series_checked=len(readiness), series_excluded=len(short), series_with_gaps=len(gappy), covariates_filled=len(bad_cov), extreme_days_kept=n_ext)

In [ ]:
# 🧰 Feature-engineering helpers used by the plots below
GROUP_OF = meta.set_index("item_id")["group"]


def indexed(panel):
    """Each series divided by its own mean, so series of different size can share an axis."""
    return panel.assign(group=panel.item_id.map(GROUP_OF),
                        idx=panel.value / panel.groupby("item_id")["value"].transform("mean"))


def eta2(v, key):
    """Share of variance explained by the group means of `key` (0 = no pattern, 1 = fully periodic)."""
    m = v.groupby(key).transform("mean")
    tot = ((v - v.mean()) ** 2).sum()
    return float(1 - ((v - m) ** 2).sum() / tot) if tot > 0 else 0.0


def seasonality_strength(panel):
    rows = []
    for item, g in panel.groupby("item_id"):
        v = np.log1p(g["value"].clip(lower=0)).reset_index(drop=True)
        t = g["timestamp"].reset_index(drop=True)
        v = v - v.rolling(SEASON * 26, center=True, min_periods=SEASON * 4).mean().bfill().ffill()   # remove trend
        rows.append({"item_id": item, "day of week": eta2(v, t.dt.dayofweek), "day of month": eta2(v, t.dt.day),
                     "month of year": eta2(v, t.dt.month)})
    return pd.DataFrame(rows).set_index("item_id")


def acf(x, max_lag):
    x = np.asarray(x, float) - np.mean(x)
    d = float(np.dot(x, x)) or 1.0
    return np.array([np.dot(x[:-k], x[k:]) / d for k in range(1, max_lag + 1)])


def decompose(g):
    """Multiplicative split: trend × weekly pattern × day-of-month pattern × remainder."""
    v = g.set_index("timestamp")["value"].astype(float)
    trend = v.rolling(SEASON * 13, center=True, min_periods=SEASON * 4).mean().bfill().ffill().clip(lower=1e-9)
    r1 = v / trend
    weekly = r1.groupby(r1.index.dayofweek).transform("mean").clip(lower=1e-9)
    r2 = r1 / weekly
    monthly = r2.groupby(r2.index.day).transform("mean").clip(lower=1e-9)
    return pd.DataFrame({"observed": v, "trend": trend, "weekly pattern": weekly,
                         "day-of-month pattern": monthly, "remainder": r2 / monthly})


def engineer_features(panel):
    """Lag, rolling and calendar features per series. Every lag is shifted, so nothing looks ahead."""
    out = []
    for item, g in panel.sort_values("timestamp").groupby("item_id"):
        f = pd.DataFrame({"item_id": item, "timestamp": g["timestamp"].values, "y": g["y"].values})
        y = f["y"]
        for l in (1, SEASON, 2 * SEASON, MONTH_LAG):
            f[f"lag_{l}"] = y.shift(l)
        f[f"roll_mean_{SEASON}"] = y.shift(1).rolling(SEASON).mean()
        f[f"roll_mean_{4 * SEASON}"] = y.shift(1).rolling(4 * SEASON).mean()
        f[f"roll_std_{4 * SEASON}"] = y.shift(1).rolling(4 * SEASON).std()
        f["same_weekday_mean_4"] = sum(y.shift(SEASON * k) for k in range(1, 5)) / 4
        f["dow"] = f["timestamp"].dt.dayofweek
        f["dom"] = f["timestamp"].dt.day
        f["month"] = f["timestamp"].dt.month
        f["days_to_month_end"] = f["timestamp"].dt.days_in_month - f["timestamp"].dt.day
        for c in KNOWN:
            f[c] = g[c].values
        for c in PAST:
            f[f"{c}_lag1"] = pd.Series(g[c].values).shift(1).values      # observed only up to yesterday
        out.append(f)
    return pd.concat(out, ignore_index=True).dropna()


def feature_family(name):
    if name.startswith("lag_") or name == "same_weekday_mean_4":
        return "lags"
    if name.startswith("roll_"):
        return "rolling statistics"
    if any(name.startswith(p) for p in PAST):
        return "market drivers"
    return "calendar"


FAMILY_COLORS = {"lags": BLUE, "rolling statistics": TEAL, "calendar": ORANGE, "market drivers": RED}

In [ ]:
# 📏 Model scale: each series as an index of its own recent average
# (errors become comparable between a small and a large series; forecasts are converted back for the dashboards)
panel = panel.sort_values(["item_id", "timestamp"]).reset_index(drop=True)
_in_history = panel.groupby("item_id").cumcount(ascending=False) >= HORIZON      # scale never sees the hold-out
SCALE = (panel[_in_history].groupby("item_id")["value"]
         .apply(lambda s: s.tail(SCALE_WINDOW).abs().mean()).clip(lower=1e-9).rename("scale"))
panel["y"] = panel["value"] / panel["item_id"].map(SCALE)

sample = meta.groupby("group").head(2).item_id
fig = make_subplots(rows=1, cols=2, subplot_titles=(f"Raw level ({UNIT}, log axis)", "Indexed to own recent average"))
for item in sample:
    g = panel[panel.item_id == item].set_index("timestamp").resample("W").mean(numeric_only=True)
    c = GROUP_COLORS[GROUP_OF[item]]
    fig.add_trace(go.Scatter(x=g.index, y=g["value"].clip(lower=1e-6), name=item, line=dict(color=c, width=1.5)), 1, 1)
    fig.add_trace(go.Scatter(x=g.index, y=g["y"], line=dict(color=c, width=1.5), showlegend=False), 1, 2)
fig.update_yaxes(type="log", row=1, col=1)
fig.update_layout(template="plotly_white", height=420, title="Scaling · why we index before modelling (weekly means)")
fig.show()

In [ ]:
# 🗓️ Seasonal profiles and how strong each pattern is, per series
d = indexed(panel)
fig = make_subplots(rows=1, cols=3, subplot_titles=("Day of week", "Day of month", "Month of year"))
for j, (key, labels) in enumerate([(d.timestamp.dt.dayofweek, DOW), (d.timestamp.dt.day, None), (d.timestamp.dt.month, MON)], 1):
    prof = d.groupby(["group", key.rename("k")])["idx"].mean().reset_index()
    for gname, gg in prof.groupby("group"):
        fig.add_trace(go.Scatter(x=[labels[k] if labels else k for k in gg.k], y=gg.idx, name=gname, mode="lines+markers",
                                 line=dict(color=GROUP_COLORS[gname], width=2), showlegend=j == 1), 1, j)
fig.update_layout(template="plotly_white", height=400, title="Average profile by group (1.0 = the series' own mean)",
                  legend=dict(orientation="h", y=-0.15))
fig.show()

strength = seasonality_strength(panel)
order = meta.sort_values("group").item_id
fig = px.imshow(strength.loc[order].T, color_continuous_scale="Blues", zmin=0, zmax=1, aspect="auto",
                title="Seasonality strength · share of variance explained by each calendar cycle")
fig.update_layout(template="plotly_white", height=360, xaxis_tickangle=-60)
fig.show()

In [ ]:
# 🔥 The payment calendar in one picture: average flow by business day of the month
d = indexed(panel)
heat = d[d.bday_of_month <= 22].groupby(["item_id", "bday_of_month"])["idx"].mean().unstack()
heat = heat.loc[meta.item_id]
fig = px.imshow(heat, color_continuous_scale="Blues", zmin=0, zmax=4, aspect="auto",
                labels=dict(x="business day of the month", y="", color="× own mean"),
                title="Each line has its own payment rhythm (1.0 = an average day for that line)")
fig.update_layout(template="plotly_white", height=620)
fig.show()

In [ ]:
# 🔁 Autocorrelation (which lags carry signal) and a decomposition of one series
reps = meta.groupby("group").head(1).item_id.tolist()
max_lag = SEASON * 9
fig = make_subplots(rows=1, cols=len(reps), subplot_titles=reps, shared_yaxes=True)
for j, item in enumerate(reps, 1):
    a = acf(panel.loc[panel.item_id == item, "y"], max_lag)
    lags = np.arange(1, max_lag + 1)
    fig.add_trace(go.Bar(x=lags, y=a, marker_color=[ORANGE if l % SEASON == 0 else GREY for l in lags], showlegend=False), 1, j)
fig.update_layout(template="plotly_white", height=360,
                  title=f"Autocorrelation · orange bars are multiples of the weekly cycle ({SEASON} steps)")
fig.show()

dec = decompose(panel[panel.item_id == DECOMPOSE_ITEM]).iloc[-SEASON * 60:]
fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.03, subplot_titles=list(dec.columns))
for i, c in enumerate(dec.columns, 1):
    fig.add_trace(go.Scatter(x=dec.index, y=dec[c], line=dict(color=[DARK, BLUE, ORANGE, TEAL, GREY][i - 1], width=1.4),
                             showlegend=False), i, 1)
fig.update_layout(template="plotly_white", height=820, title=f"{DECOMPOSE_ITEM} · observed = trend × weekly × day-of-month × remainder")
fig.show()

In [ ]:
# 🎛️ Calendar covariates: catalogue, and the uplift each flag carries by group
catalogue = pd.DataFrame([{"feature": c, "family": "calendar (known in advance)", "description": COV_DOC.get(c, "")} for c in KNOWN]
                         + [{"feature": c, "family": "market driver (observed up to today)", "description": COV_DOC.get(c, "")} for c in PAST])
display(catalogue.style.set_caption("Covariates handed to the models").hide(axis="index"))

flags = [c for c in KNOWN if panel[c].dropna().isin([0, 1]).all()]
d = indexed(panel)
rows = {}
for c in flags:
    on = d[d[c] == 1].groupby("group")["idx"].mean()
    off = d[d[c] == 0].groupby("group")["idx"].mean()
    rows[c] = (on / off - 1) * 100
uplift = pd.DataFrame(rows).T
fig = px.imshow(uplift, color_continuous_scale="RdBu_r", zmin=-100, zmax=100, text_auto=".0f", aspect="auto",
                title="Average change in flow when a calendar flag is on (% vs days when it is off)")
fig.update_layout(template="plotly_white", height=40 * len(flags) + 160)
fig.show()

In [ ]:
# 🧪 Which engineered features matter? A gradient-boosting probe on a time-ordered hold-out
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error

feats = engineer_features(panel)
cut = feats.timestamp.quantile(0.85)
tr, te = feats[feats.timestamp <= cut], feats[feats.timestamp > cut]
cols = [c for c in feats.columns if c not in ("item_id", "timestamp", "y")]
fam = {c: feature_family(c) for c in cols}
print(f"{len(cols)} engineered features · {len(tr):,} training rows · {len(te):,} hold-out rows (after {cut:%d %b %Y})")
display(feats[["item_id", "timestamp", "y"] + cols[:8]].tail(3))

# Ablation: what each feature family adds
steps, used, maes = ["calendar", "lags", "rolling statistics", "market drivers"], [], {}
for s in steps:
    add = [c for c in cols if fam[c] == s]
    if not add:
        continue
    used += add
    m = HistGradientBoostingRegressor(max_iter=150, random_state=0).fit(tr[used], tr["y"])
    maes["+ " + s if len(maes) else s] = mean_absolute_error(te["y"], m.predict(te[used]))
naive = mean_absolute_error(te["y"], te[f"lag_{SEASON}"])

probe = te.sample(min(6000, len(te)), random_state=0)
imp = permutation_importance(m, probe[used], probe["y"], n_repeats=3, random_state=0, scoring="neg_mean_absolute_error")
imp = pd.Series(imp.importances_mean, index=used).sort_values().tail(18)

fig = make_subplots(rows=1, cols=2, column_widths=[0.4, 0.6],
                    subplot_titles=("Hold-out error as feature families are added", "Permutation importance (top 18)"))
fig.add_trace(go.Bar(x=["seasonal naive"] + list(maes), y=[naive] + list(maes.values()),
                     marker_color=[GREY] + [FAMILY_COLORS[s.replace("+ ", "")] for s in maes],
                     text=[f"{v:.3f}" for v in [naive] + list(maes.values())], textposition="outside", showlegend=False), 1, 1)
fig.add_trace(go.Bar(x=imp.values, y=imp.index, orientation="h", marker_color=[FAMILY_COLORS[fam[c]] for c in imp.index],
                     showlegend=False), 1, 2)
for f_, c_ in FAMILY_COLORS.items():
    fig.add_trace(go.Bar(x=[None], y=[None], marker_color=c_, name=f_), 1, 2)
fig.update_yaxes(title_text="MAE (indexed units)", row=1, col=1)
fig.update_layout(template="plotly_white", height=560, title="Feature engineering pays: lags, rolling statistics and calendar",
                  legend=dict(orientation="h", y=-0.1))
fig.show()

In [ ]:
# 📌 Data preparation: conclusions
display(Markdown(f"""
### Data preparation: conclusions
- **{FACTS['data']['series']} series, {FACTS['data']['panel_rows']:,.0f} rows**, from {FACTS['data']['first_date']} to {FACTS['data']['last_date']}, on a regular calendar grid.
- The readiness agent checked {FACTS['preparation']['series_checked']:.0f} series: {FACTS['preparation']['series_excluded']:.0f} excluded, {FACTS['preparation']['covariates_filled']:.0f} covariates filled, {FACTS['preparation']['extreme_days_kept']:.0f} extreme days kept as real events.
- Each series is **scaled to its own average**, so large and small series are comparable and one model serves all of them.
- **{len(KNOWN)} calendar covariates** are known in advance and go to the model for the future window; {len(PAST)} market drivers are known only for the past.
- Every agent step and approval is in the audit trail at the end of the notebook.
"""))
record("features", scaling="each series divided by its own trailing average", calendar_covariates=", ".join(KNOWN[:8]), market_drivers=", ".join(PAST) or "none",
       seasonal_period=SEASON, horizon_steps=HORIZON)

# 4 · Model training

## 4.1 AutoML race on rolling backtests

`TimeSeriesLeaderboard` scores every model on the last three 20-business-day windows of history, never on data it was fitted on. Foundation models run **zero-shot**: they have not been trained on Treasury data. **MASE** below 1 beats a seasonal-naive forecast; **WQL** scores the whole predictive distribution.

In [ ]:
# 🏁 Hold out the final horizon, race the models on rolling backtests of the rest
_past = {"past_covariates": PAST} if PAST else {}
schema = TimeSeriesSchema(target="y", item_id="item_id")
schema_cov = TimeSeriesSchema(target="y", item_id="item_id", known_covariates=KNOWN, **_past)
model_df = panel[["item_id", "timestamp", "y", *KNOWN, *PAST]]
history, actual = split_horizon(model_df, schema_cov, prediction_length=HORIZON)
future_cov = actual[["item_id", "timestamp", *KNOWN]]
history_y = history[["item_id", "timestamp", "y"]]
value_hist = panel.merge(history[["item_id", "timestamp"]], on=["item_id", "timestamp"])
value_act = panel.merge(actual[["item_id", "timestamp"]], on=["item_id", "timestamp"])
fp = {"prediction_length": HORIZON, "quantile_levels": QUANTILES}

board = TimeSeriesLeaderboard(history_y, schema, forecast_params=fp, windows=3)
board.add_models([m for m in BASELINES if m != "TabularTS-GBM"] + FOUNDATION_MODELS)
board.add_model("TabularTS-GBM", model_params=GBM_PARAMS)
t0 = time.time()
ranking = board.run(rank_by="mase", display=False)
print(f"{history.item_id.nunique()} series · {len(history):,} history rows · hold-out {HORIZON} steps · leaderboard in {time.time() - t0:.0f}s")
display(ranking)

ok = ranking[ranking["Status"].eq("ok")] if "Status" in ranking else ranking
fig = make_subplots(rows=1, cols=2, subplot_titles=("MASE (lower is better)", "WQL (lower is better)"))
for i, metric in enumerate(["mase", "wql"], start=1):
    o = ok.sort_values(metric)
    fig.add_trace(go.Bar(y=o["Model"], x=o[metric], orientation="h", text=o[metric].round(3), textposition="outside",
                         marker_color=[BLUE if m in FOUNDATION_MODELS else GREY for m in o["Model"]], showlegend=False), 1, i)
fig.add_vline(x=1.0, line_dash="dot", line_color=DARK, row=1, col=1)
fig.update_layout(template="plotly_white", height=420,
                  title="Rolling-backtest leaderboard · blue = foundation model (zero-shot), grey = baseline")
fig.show()

## 4.2 Hyperparameter tuning
Two kinds of knob are swept on an inner backtest window carved from the history (the final hold-out is not touched): the **tree forecaster's boosting rounds**, and the **context length** given to the selected model, which is the main tuning lever for a zero-shot foundation model.

In [ ]:
# 🔬 Sweeps on an inner backtest window
inner_hist, inner_act = split_horizon(history, schema_cov, prediction_length=HORIZON)
inner_y = inner_hist[["item_id", "timestamp", "y"]]

def quick_score(model, df, params=None):
    p = TimeSeriesPipeline(model, forecast_params=fp, tuning_params={"device": DEVICE}, model_params=dict(params or {}))
    t0 = time.time(); p.fit(df, schema); m = p.evaluate(inner_act)
    return {"mase": m["mase"], "wql": m["wql"], "seconds": time.time() - t0}

ROUNDS = [30, 60] if FAST else [30, 60, 120, 240]
gbm_sweep = pd.DataFrame([{"max_iter": r, **quick_score("TabularTS-GBM", inner_y, {"max_iter": r})} for r in ROUNDS])
CONTEXTS = [MONTH_LAG * 4, SCALE_WINDOW] if FAST else [MONTH_LAG * 3, MONTH_LAG * 6, SCALE_WINDOW, SCALE_WINDOW * 2, None]
ctx_rows = []
for n_ctx in CONTEXTS:
    d = inner_y if n_ctx is None else inner_y.groupby("item_id").tail(n_ctx)
    ctx_rows.append({"context": "all history" if n_ctx is None else f"{n_ctx} steps", **quick_score(PRIMARY, d, PRIMARY_PARAMS)})
ctx_sweep = pd.DataFrame(ctx_rows)
GBM_PARAMS = {"max_iter": int(gbm_sweep.sort_values("mase").iloc[0]["max_iter"])}
BEST_CONTEXT = ctx_sweep.sort_values("mase").iloc[0]["context"]

fig = make_subplots(rows=1, cols=2, specs=[[{"secondary_y": True}, {"secondary_y": True}]],
                    subplot_titles=("TabularTS-GBM · boosting rounds", f"{PRIMARY} · context length"))
for col, (df_, x) in enumerate(((gbm_sweep, "max_iter"), (ctx_sweep, "context")), 1):
    fig.add_trace(go.Scatter(x=df_[x].astype(str), y=df_.mase, mode="lines+markers", name="MASE", line=dict(color=BLUE, width=3), showlegend=col == 1), 1, col)
    fig.add_trace(go.Bar(x=df_[x].astype(str), y=df_.seconds, name="seconds", marker_color="rgba(154,160,168,0.35)", showlegend=col == 1), 1, col, secondary_y=True)
fig.update_yaxes(title_text="MASE (lower is better)", row=1, col=1); fig.update_yaxes(title_text="seconds", secondary_y=True, row=1, col=2)
fig.update_layout(template="plotly_white", height=420, title=f"Hyperparameter sweeps · best rounds {GBM_PARAMS['max_iter']} · best context: {BEST_CONTEXT}")
fig.show()
audit("automl", "hyperparameter sweep", f"GBM rounds {ROUNDS} → {GBM_PARAMS['max_iter']}; {PRIMARY} context → {BEST_CONTEXT}")

## 4.3 Model selection
**Rule:** the production model must (1) carry a licence cleared for commercial use, (2) accept calendar covariates that are known in advance, and (3) have the lowest backtest MASE among models that meet 1 and 2. The leaderboard winner on target-only data is shown next to it as a challenger for section 9.

In [ ]:
# 🏆 Selection rule applied to the leaderboard
okb = (ranking[ranking["Status"].eq("ok")] if "Status" in ranking else ranking).sort_values("mase").reset_index(drop=True)
LB_WINNER = str(okb.iloc[0]["Model"])
mase_of = okb.set_index("Model")["mase"].to_dict()
READS_COVARIATES = {"Chronos2", "TabularTS-GBM"}
elig = [m for m in okb["Model"] if m in READS_COVARIATES]
CHAMPION = PRIMARY if PRIMARY in elig or not elig else elig[0]
tbl = okb[["Model", "mase", "wql"]].assign(reads_covariates=okb["Model"].isin(READS_COVARIATES),
                                             role=[("selected" if m == CHAMPION else "challenger" if m == LB_WINNER else "") for m in okb["Model"]])
fig = go.Figure(go.Bar(x=tbl.mase, y=tbl.Model, orientation="h", text=tbl.mase.round(3), textposition="outside",
                       marker_color=[BLUE if m == CHAMPION else ORANGE if m == LB_WINNER else GREY for m in tbl.Model]))
fig.add_vline(x=1.0, line_dash="dot", line_color=DARK, annotation_text="seasonal-naive level")
fig.update_layout(template="plotly_white", height=380, yaxis=dict(autorange="reversed"),
                  title=f"Selection · blue = selected ({CHAMPION}), orange = best on target-only backtest ({LB_WINNER})", xaxis_title="backtest MASE")
fig.show()
display(tbl.style.hide(axis="index").format({"mase": "{:.3f}", "wql": "{:.3f}"}))
audit("automl", "model selection", f"{len(okb)} models backtested; lowest MASE {LB_WINNER} ({mase_of[LB_WINNER]:.3f})", f"SELECTED {CHAMPION}")
human_gate(f"Proceed with {CHAMPION} as the production forecaster?", True)
record("selection", candidates_raced=len(ranking), candidates_succeeded=len(okb), selected_model=CHAMPION, selection_metric="rolling-backtest MASE",
       selected_backtest_mase=float(mase_of.get(CHAMPION, np.nan)), leaderboard_winner=LB_WINNER, leaderboard_winner_mase=float(mase_of[LB_WINNER]),
       seasonal_naive_mase=float(mase_of.get("SeasonalNaive", np.nan)),
       selection_rule="commercial licence; reads known-in-advance covariates; lowest backtest MASE among those", tuned_context=str(BEST_CONTEXT))
record("training", adaptation="zero-shot (no gradient training)" if CHAMPION in FOUNDATION_MODELS else "trained on the history",
       history_rows=len(history), backtest_windows=3, device=GPU)

## 4.4 Fit the selected model with covariates, then calibrate its bands

In [ ]:
# 🔮 Forecast with and without covariates, then calibrate the intervals
plain = TimeSeriesPipeline(PRIMARY, forecast_params=fp, license_mode="commercial", tuning_params={"device": DEVICE},
                           model_params=dict(PRIMARY_PARAMS))
plain.fit(history_y, schema)
m_plain = plain.evaluate(actual)

pipe = TimeSeriesPipeline(PRIMARY, forecast_params=fp, license_mode="commercial", tuning_params={"device": DEVICE},
                          model_params=dict(PRIMARY_PARAMS))
pipe.fit(history, schema_cov, future_df=future_cov)
m_cov = pipe.evaluate(actual, forecast=pipe.predict())

pipe.calibrate(windows=4)
forecast_cal = pipe.predict()
m_cal = pipe.evaluate(actual, forecast=forecast_cal)

KEYS = [k for k in ("mase", "wql", "smape", "coverage_80", "coverage_90") if k in m_cov]
display(pd.DataFrame({"target only": m_plain, "with covariates": m_cov, "with covariates, calibrated": m_cal}).loc[KEYS])
gain = (1 - m_cov["mase"] / m_plain["mase"]) * 100
display(Markdown(f"**Covariates change MASE by {gain:+.1f}%** on the hold-out ({PRIMARY})."))

fc = forecast_cal.to_pandas()
QCOLS = ["point"] + [str(q) for q in QUANTILES]
fc[QCOLS] = fc[QCOLS].mul(fc.item_id.map(SCALE), axis=0)          # back to business units

## 4.5 Global explainability
Which inputs does the selected model rely on? Each calendar covariate is removed in turn, the model is re-run on the inner backtest window, and the change in error is the covariate's importance. (The feature-family probe in section 3 answers the same question for engineered lags and rolling statistics.)

In [ ]:
# 🔍 Covariate ablation on the inner backtest window
inner_future = inner_act[["item_id", "timestamp", *KNOWN]]
def score_with(known):
    sch = TimeSeriesSchema(target="y", item_id="item_id", known_covariates=list(known), **_past) if (known or PAST) else schema
    cols = ["item_id", "timestamp", "y", *known, *PAST]
    p = TimeSeriesPipeline(PRIMARY, forecast_params=fp, tuning_params={"device": DEVICE}, model_params=dict(PRIMARY_PARAMS))
    p.fit(inner_hist[cols], sch, future_df=inner_future[["item_id", "timestamp", *known]] if known else None)
    return p.evaluate(inner_act[cols], forecast=p.predict())["mase"]

full_mase = score_with(KNOWN)
ABLATE = KNOWN[:2] if FAST else KNOWN[:8]
abl = pd.Series({k: score_with([c for c in KNOWN if c != k]) - full_mase for k in ABLATE}).sort_values()
none_mase = score_with([])
fig = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4], subplot_titles=("Rise in MASE when one covariate is removed", "All covariates vs none"))
fig.add_trace(go.Bar(x=abl.values, y=abl.index, orientation="h", marker_color=[BLUE if v > 0 else GREY for v in abl.values], showlegend=False), 1, 1)
fig.add_trace(go.Bar(x=["no covariates", "all covariates"], y=[none_mase, full_mase], marker_color=[GREY, BLUE], showlegend=False,
                     text=[f"{none_mase:.3f}", f"{full_mase:.3f}"], textposition="outside"), 1, 2)
fig.update_layout(template="plotly_white", height=440, title=f"Global explanation · {PRIMARY} · positive bar = the model was using that covariate")
fig.show()
record("explainability", method="leave-one-covariate-out on an inner backtest window; feature-family probe for engineered features",
       most_useful_covariates=", ".join(abl.sort_values(ascending=False).head(3).index), mase_without_covariates=float(none_mase), mase_with_covariates=float(full_mase))

## 4.6 Ensembles and series similarity

In [ ]:
# 🤝 Greedy ensemble, and series similarity from foundation-model embeddings
members = [m if m != "TabularTS-GBM" else {"model_name": m, "model_params": GBM_PARAMS} for m in BASELINES + FOUNDATION_MODELS]
ensemble = TimeSeriesEnsemble(members, ensemble_strategy="greedy_selection",
                              forecast_params=fp, verbose=False)
ensemble.fit(history_y, schema)
display(ensemble.get_leaderboard())
m_ens = ensemble.evaluate(actual)
display(pd.Series({k: m_ens[k] for k in ("mase", "wql") if k in m_ens}, name="ensemble hold-out").to_frame())

if EMBEDDER:
    emb = TimeSeriesPipeline(EMBEDDER, task_type="embedding", tuning_params={"device": DEVICE}).fit(history_y, schema).predict()
    sim = pd.DataFrame(emb.similarity(), index=list(emb.item_ids), columns=list(emb.item_ids))
    order = meta.sort_values("group").item_id
    fig = px.imshow(sim.loc[order, order], color_continuous_scale="Blues", zmax=1,
                    title=f"Cosine similarity of {EMBEDDER} embeddings · {emb.dim}-dim per series")
    fig.update_layout(height=720, template="plotly_white"); fig.show()
else:
    print("Embeddings need a foundation model (skipped in fast mode).")

if RUN_FINETUNE and not FAST:
    tuned = TimeSeriesPipeline("ChronosBolt", tuning_strategy="peft", forecast_params=fp,
                               tuning_params={"epochs": 2, "steps_per_epoch": 50, "device": DEVICE}).fit(history_y, schema)
    zero = TimeSeriesPipeline("ChronosBolt", forecast_params=fp, tuning_params={"device": DEVICE}).fit(history_y, schema)
    display(pd.DataFrame({"zero-shot": zero.evaluate(actual), "LoRA, 2 epochs": tuned.evaluate(actual)}).loc[["mase", "wql"]])

# 5 · Model testing
Everything in this section uses the final hold-out window, which no step above has scored against.

## 5.1 Forecast vs actual, series by series

In [ ]:
# 🌀 Interactive fan chart: pick a series from the dropdown
tail = value_hist[value_hist.timestamp > value_hist.timestamp.max() - pd.Timedelta(days=150)]
items = list(meta.item_id)
fig = go.Figure()
for k, item in enumerate(items):
    h = tail[tail.item_id == item]; a = value_act[value_act.item_id == item]; f = fc[fc.item_id == item]
    vis = k == 0
    fig.add_trace(go.Scatter(x=h.timestamp, y=h.value, name="history", line=dict(color=DARK, width=1.5), visible=vis))
    fig.add_trace(go.Scatter(x=pd.concat([f.timestamp, f.timestamp[::-1]]), y=pd.concat([f["0.95"], f["0.05"][::-1]]),
                             fill="toself", fillcolor="rgba(47,91,234,0.12)", line=dict(width=0), name="90% band", visible=vis))
    fig.add_trace(go.Scatter(x=pd.concat([f.timestamp, f.timestamp[::-1]]), y=pd.concat([f["0.9"], f["0.1"][::-1]]),
                             fill="toself", fillcolor="rgba(47,91,234,0.25)", line=dict(width=0), name="80% band", visible=vis))
    fig.add_trace(go.Scatter(x=f.timestamp, y=f["0.5"], name="median forecast", line=dict(color=BLUE, width=2.5), visible=vis))
    fig.add_trace(go.Scatter(x=a.timestamp, y=a.value, name="actual", mode="markers", marker=dict(color=ORANGE, size=6), visible=vis))
n = 5
buttons = [dict(label=item, method="update",
                args=[{"visible": [j // n == k for j in range(n * len(items))]}, {"title": f"{item} · forecast"}])
           for k, item in enumerate(items)]
fig.update_layout(template="plotly_white", height=520, title=f"{items[0]} · forecast", yaxis_title=f"{UNIT} per day",
                  updatemenus=[dict(buttons=buttons, x=1.0, xanchor="right", y=1.15, direction="down")],
                  legend=dict(orientation="h", y=-0.15))
fig.show()

# Accuracy by series: where the model is strong and where it is not
per = fc.merge(value_act[["item_id", "timestamp", "value"]], on=["item_id", "timestamp"])
acc = per.groupby("item_id").apply(lambda g: pd.Series({
    "wape_%": (g["0.5"] - g["value"]).abs().sum() / max(g["value"].abs().sum(), 1e-9) * 100,
    "inside_90_%": g["value"].between(g["0.05"], g["0.95"]).mean() * 100})).reset_index()
acc["group"] = acc.item_id.map(GROUP_OF)
fig = px.bar(acc.sort_values("wape_%"), x="item_id", y="wape_%", color="group", color_discrete_map=GROUP_COLORS,
             title="Hold-out error by series (weighted absolute % error of the median forecast)")
fig.update_layout(template="plotly_white", height=430, xaxis_tickangle=-60, xaxis_title="")
fig.show()

## 5.2 Hold-out dashboard

In [ ]:
# 📊 Test dashboard on the hold-out window
per = fc.merge(value_act[["item_id", "timestamp", "value"]], on=["item_id", "timestamp"]).sort_values(["item_id", "timestamp"])
per["step"] = per.groupby("item_id").cumcount() + 1
per["group"] = per.item_id.map(GROUP_OF)
per["abs_err"] = (per["0.5"] - per["value"]).abs()
by_step = per.groupby("step").apply(lambda g: pd.Series({"wape": g.abs_err.sum() / max(g["value"].abs().sum(), 1e-9) * 100,
                                                         "inside_90": g["value"].between(g["0.05"], g["0.95"]).mean() * 100,
                                                         "width": ((g["0.95"] - g["0.05"]) / g.item_id.map(SCALE)).mean()}))
qcal = pd.Series({q: (per["value"] <= per[str(q)]).mean() for q in QUANTILES})
COVK = [k for k in m_cal if k.startswith("coverage")]
fig = make_subplots(rows=2, cols=3, vertical_spacing=0.16, specs=[[{}, {}, {}], [{}, {}, {"secondary_y": True}]],
                    subplot_titles=("Error by forecast step (WAPE %)", "Quantile calibration: promised vs observed", "Error by segment (WAPE % per series)",
                                    "Forecast vs actual (all series-days, log scale)", "Accuracy: target only → covariates → calibrated", "Band coverage and width by step"))
fig.add_trace(go.Scatter(x=by_step.index, y=by_step.wape, mode="lines+markers", line=dict(color=BLUE, width=3), showlegend=False), 1, 1)
fig.add_trace(go.Scatter(x=qcal.index, y=qcal.values, mode="lines+markers", line=dict(color=TEAL, width=3), showlegend=False), 1, 2)
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], line=dict(color=GREY, dash="dot"), showlegend=False), 1, 2)
seg = per.groupby(["group", "item_id"]).apply(lambda g: g.abs_err.sum() / max(g["value"].abs().sum(), 1e-9) * 100).reset_index(name="wape")
for gname, g in seg.groupby("group"):
    fig.add_trace(go.Box(y=g.wape, name=gname, marker_color=GROUP_COLORS.get(gname, GREY), boxpoints="all", showlegend=False), 1, 3)
pos = per[(per["value"] > 0) & (per["0.5"] > 0)]
fig.add_trace(go.Scattergl(x=pos["value"], y=pos["0.5"], mode="markers", marker=dict(size=3, color=BLUE, opacity=0.35), showlegend=False), 2, 1)
lim = [pos["value"].quantile(0.01), pos["value"].quantile(0.999)]
fig.add_trace(go.Scatter(x=lim, y=lim, line=dict(color=DARK, dash="dot"), showlegend=False), 2, 1)
fig.update_xaxes(type="log", row=2, col=1); fig.update_yaxes(type="log", row=2, col=1)
for name, mm, col in (("target only", m_plain, GREY), ("with covariates", m_cov, ORANGE), ("calibrated", m_cal, BLUE)):
    fig.add_trace(go.Bar(x=["MASE", "WQL ×10"], y=[mm["mase"], mm["wql"] * 10], name=name, marker_color=col), 2, 2)
fig.add_trace(go.Scatter(x=by_step.index, y=by_step.inside_90, name="inside 90% band (%)", line=dict(color=TEAL, width=3)), 2, 3)
fig.add_trace(go.Scatter(x=by_step.index, y=by_step.width, name="band width (× average day)", line=dict(color=ORANGE, dash="dot")), 2, 3, secondary_y=True)
fig.update_layout(template="plotly_white", height=820, barmode="group", legend=dict(orientation="h", y=-0.08),
                  title=f"Hold-out evaluation · {PRIMARY} · {per.item_id.nunique()} series × {HORIZON} steps")
fig.show()
kp = [("MASE", m_cal["mase"]), ("WQL", m_cal["wql"]), ("sMAPE", m_cal.get("smape", np.nan)), ("WAPE %", per.abs_err.sum() / per["value"].abs().sum() * 100),
      ("inside 90% band %", per["value"].between(per["0.05"], per["0.95"]).mean() * 100), ("inside 80% band %", per["value"].between(per["0.1"], per["0.9"]).mean() * 100)]
fig = make_subplots(rows=1, cols=len(kp), specs=[[{"type": "indicator"}] * len(kp)])
for i, (k, v) in enumerate(kp, 1):
    fig.add_trace(go.Indicator(mode="number", value=v, title={"text": k}, number={"valueformat": ",.3f" if v < 10 else ",.1f", "font": {"size": 40}}), 1, i)
fig.update_layout(height=200, margin=dict(t=50, b=10)); fig.show()
record("performance", evaluated_on=f"final {HORIZON}-step hold-out, never used for selection", series=int(per.item_id.nunique()), mase=m_cal["mase"], wql=m_cal["wql"],
       smape=m_cal.get("smape", np.nan), wape_pct=kp[3][1], inside_90_band_pct=kp[4][1], inside_80_band_pct=kp[5][1],
       mase_target_only=m_plain["mase"], mase_with_covariates=m_cov["mase"])

## 5.3 Business view · Treasury dashboard: net operating cash position

In [ ]:
# 🧭 Inflows minus outflows over the next 20 business days
sign = fc.item_id.map(lambda i: 1 if GROUP_OF[i] == "Inflow" else -1)
per = fc.merge(value_act[["item_id", "timestamp", "value"]], on=["item_id", "timestamp"])
per["sign"] = per.item_id.map(lambda i: 1 if GROUP_OF[i] == "Inflow" else -1)
net = per.assign(f=per["0.5"] * per.sign, a=per.value * per.sign).groupby("timestamp")[["f", "a"]].sum()
grp = per.assign(group=per.item_id.map(GROUP_OF)).groupby(["group", "timestamp"])[["0.5", "value"]].sum().reset_index()
tot = per.groupby("item_id").agg(forecast=("0.5", "sum"), actual=("value", "sum"))
tot["error_%"] = (tot.forecast / tot.actual.replace(0, np.nan) - 1) * 100
tot = tot.loc[meta.item_id]

fig = make_subplots(rows=2, cols=4, row_heights=[0.28, 0.72], vertical_spacing=0.13,
                    specs=[[{"type": "indicator"}] * 4, [{"colspan": 2}, None, {"colspan": 2}, None]],
                    subplot_titles=("", "", "", "", "Cumulative net operating cash flow (USD bn)", "20-day totals by line: forecast vs actual (USD bn)"))
inflow, outflow = tot.forecast[meta.group.values == "Inflow"].sum(), tot.forecast[meta.group.values == "Outflow"].sum()
kpis = [("Forecast inflow<br>20 days, USD bn", inflow / 1e3), ("Forecast outflow<br>20 days, USD bn", outflow / 1e3),
        ("Forecast net<br>USD bn", (inflow - outflow) / 1e3), ("Forecast error<br>hold-out MASE", m_cal["mase"])]
for i, (title, v) in enumerate(kpis, 1):
    fig.add_trace(go.Indicator(mode="number", value=v, title={"text": title},
                               number={"valueformat": ",.2f" if abs(v) < 50 else ",.0f", "font": {"size": 46}}), 1, i)
fig.add_trace(go.Scatter(x=net.index, y=net.f.cumsum() / 1e3, name="forecast (median)", line=dict(color=BLUE, width=3)), 2, 1)
fig.add_trace(go.Scatter(x=net.index, y=net.a.cumsum() / 1e3, name="actual", mode="lines+markers", line=dict(color=DARK, dash="dot")), 2, 1)
fig.add_trace(go.Bar(y=tot.index, x=tot.forecast / 1e3, orientation="h", name="forecast", marker_color=BLUE), 2, 3)
fig.add_trace(go.Bar(y=tot.index, x=tot.actual / 1e3, orientation="h", name="actual", marker_color=GREY), 2, 3)
fig.update_layout(template="plotly_white", height=820, barmode="group", legend=dict(orientation="h", y=-0.06),
                  title=f"Operating cash outlook · {fc.timestamp.min():%d %b} – {fc.timestamp.max():%d %b %Y} · model {PRIMARY}")
fig.show()
display(tot.style.format({"forecast": "{:,.0f}", "actual": "{:,.0f}", "error_%": "{:+.1f}%"})
        .background_gradient(subset=["error_%"], cmap="RdBu_r", vmin=-40, vmax=40)
        .set_caption("20-business-day totals per line (USD mn), hold-out period"))

# 6 · Model inference in batch mode
One call forecasts every series for the whole horizon. The result is written as a long file (series, date, quantiles) that a treasury system can load directly.

In [ ]:
# 📦 Batch forecast for every series → file
t0 = time.time(); batch_fc = pipe.predict().to_pandas(); secs = time.time() - t0
batch_fc[QCOLS] = batch_fc[QCOLS].mul(batch_fc.item_id.map(SCALE), axis=0)
BATCH_OUT = "batch_forecast.csv"
batch_fc.assign(unit=UNIT, model=PRIMARY).to_csv(BATCH_OUT, index=False)
grid = (batch_fc.assign(idx=batch_fc["0.5"] / batch_fc.item_id.map(SCALE)).pivot(index="item_id", columns="timestamp", values="idx")
        .reindex(meta.sort_values("group").item_id))
fig = make_subplots(rows=1, cols=2, column_widths=[0.72, 0.28], subplot_titles=("Median forecast, indexed (1.0 = the series' average day)", f"Horizon total by segment ({UNIT})"))
fig.add_trace(go.Heatmap(z=grid.values, x=grid.columns, y=grid.index, colorscale="Blues", colorbar=dict(title="index", x=0.69)), 1, 1)
tot_ = batch_fc.assign(group=batch_fc.item_id.map(GROUP_OF)).groupby("group")["0.5"].sum()
fig.add_trace(go.Bar(x=tot_.index, y=tot_.values, marker_color=[GROUP_COLORS.get(g, GREY) for g in tot_.index], text=tot_.values, texttemplate="%{text:,.0f}", showlegend=False), 1, 2)
fig.update_layout(template="plotly_white", height=max(480, 20 * len(grid)), title=f"Batch forecast · {batch_fc.item_id.nunique()} series × {HORIZON} steps in {secs:.1f}s → {BATCH_OUT}")
fig.show()
display(batch_fc.head(6))
record("performance", batch_series=int(batch_fc.item_id.nunique()), batch_seconds=secs)
display(Markdown("**Lexsi platform note.** Batch scoring of classification and regression models on the platform is `project.model_inference(tag=..., model_name=..., compute_type=...)`. "
                 "Time-series batches run through the TabTune library call shown here, on the same managed compute."))

# 7 · Model risk

## 7.1 Uncertainty
A forecast is a range, not a number. The bands come from the model's quantiles and are then **conformally calibrated** on rolling windows, so that a 90% band holds about 90% of outcomes. The check below measures that on the hold-out.

In [ ]:
# 🎯 Uncertainty: band coverage before and after calibration, and automatic accuracy guards
cov_tab = pd.DataFrame({"with covariates": m_cov, "calibrated": m_cal}).loc[COVK] * (100 if max(m_cal[k] for k in COVK) <= 1 else 1)
nominal = [float(k.split("_")[1]) for k in COVK]
width_series = per.groupby("item_id").apply(lambda g: ((g["0.95"] - g["0.05"]) / g.item_id.map(SCALE)).mean()).sort_values()
inside_series = per.groupby("item_id").apply(lambda g: g["value"].between(g["0.05"], g["0.95"]).mean() * 100)
fig = make_subplots(rows=1, cols=3, subplot_titles=("Band coverage: promised vs measured (%)", "Share of days inside the 90% band, by series", "Band width by series (× average day)"))
fig.add_trace(go.Bar(x=[f"{n:.0f}% band" for n in nominal], y=nominal, name="promised", marker_color=GREY), 1, 1)
for name, col in (("with covariates", ORANGE), ("calibrated", TEAL)):
    fig.add_trace(go.Bar(x=[f"{n:.0f}% band" for n in nominal], y=cov_tab[name].values, name=name, marker_color=col), 1, 1)
fig.add_trace(go.Histogram(x=inside_series.values, nbinsx=15, marker_color=BLUE, showlegend=False), 1, 2)
fig.add_vline(x=90, line_dash="dot", line_color=DARK, row=1, col=2)
fig.add_trace(go.Bar(x=width_series.values, y=width_series.index, orientation="h", marker_color=ORANGE, showlegend=False), 1, 3)
fig.update_yaxes(showticklabels=False, row=1, col=3)
fig.update_layout(template="plotly_white", height=440, barmode="group", title="Uncertainty you can check: calibrated prediction bands on the hold-out", legend=dict(orientation="h", y=-0.15))
fig.show()

measured = float(cov_tab["calibrated"].iloc[-1]); promised = nominal[-1]
qs = per[[str(q) for q in QUANTILES]].to_numpy()
checks = [("no missing forecasts", bool(np.isfinite(per[QCOLS].to_numpy()).all())),
          ("quantiles are ordered (P5 ≤ P10 ≤ … ≤ P95)", bool((np.diff(qs, axis=1) >= -1e-6).mean() > 0.999)),
          ("beats the seasonal-naive forecast (MASE below 1)", bool(m_cal["mase"] < 1)),
          ("covariates do not hurt accuracy", bool(m_cov["mase"] <= m_plain["mase"] * 1.02)),
          (f"measured coverage within 10 points of the {promised:.0f}% promise", bool(abs(measured - promised) <= 10)),
          ("hold-out error within 25% of the backtest error (no selection overfit)", bool(m_cal["mase"] <= 1.25 * mase_of.get(CHAMPION, m_cal["mase"])))]
display(pd.DataFrame(checks, columns=["check", "passed"]).style.hide(axis="index").map(lambda v: f"color:{TEAL if v else RED};font-weight:600", subset=["passed"]))
record("uncertainty", method="model quantiles with conformal calibration on rolling windows", promised_coverage_pct=promised, measured_coverage_pct=measured,
       coverage_before_calibration_pct=float(cov_tab["with covariates"].iloc[-1]), average_band_width_vs_average_day=float(width_series.mean()))
record("limitations", note_1="fairness metrics do not apply to aggregate cash-flow series; consistency across segments is reported instead",
       note_2="regime changes outside the history (policy shocks, crises) widen errors until new data arrives; the monitor in section 9 is the control",
       note_3="the forecaster is zero-shot or lightly trained; it has not been validated on other institutions' data")
record("governance", frameworks_referenced="SR 11-7; SAMA model risk expectations; SAMA and NDMO data standards; CBB AI policy; SDAIA AI Ethics Principles; EU AI Act; ISO/IEC 42001",
       sanity_checks_passed=int(sum(v for _, v in checks)), sanity_checks_total=len(checks), human_approval_gates=sum(1 for a in AUDIT if a["actor"] == "human"))

## 7.2 Regulatory evidence map
Fairness in the consumer-lending sense does not apply to aggregate cash-flow series, so the table reports consistency across segments in its place.

In [ ]:
# 🏛️ Model risk: which evidence in this notebook supports which expectation
REG_MAP = pd.DataFrame([
    ("Conceptual soundness", "Problem statement, data preparation log, feature rationale, model selection rule", "SR 11-7 · SAMA model risk expectations"),
    ("Data quality and lineage", "Data profile, agent action log with human approvals, leakage checks", "SAMA Data Management · NDMO standards · PDPL"),
    ("Outcomes analysis", "Hold-out metrics, evaluation dashboards, segment performance", "SR 11-7 · Basel sound-practice principles"),
    ("Uncertainty", "Conformal prediction with measured coverage, calibration", "SR 11-7 (model limitations) · CBB AI policy"),
    ("Consistency across segments", "Error and band coverage by segment and by series", "SR 11-7 (outcomes analysis) · CBB AI policy"),
    ("Explainability", "Global drivers and per-decision explanations", "SDAIA AI Ethics Principles · EU AI Act"),
    ("Ongoing monitoring", "Drift monitoring, performance tracking, retraining with approval gate", "SR 11-7 · SAMA model risk expectations"),
    ("Documentation", "Generated model card with grounding check, audit trail", "SR 11-7 · ISO/IEC 42001"),
], columns=["Expectation", "Evidence produced in this notebook", "Frameworks it speaks to"])
display(REG_MAP.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"})
        .set_caption("Frameworks are named for orientation only. The bank's Compliance and Risk functions own the control mapping; "
                     "no clause-level compliance is claimed here."))

# 8 · Artefacts before production

## 8.1 Model card
An open-source LLM (Qwen 2.5, running on this notebook's GPU) writes the model card **only from the facts recorded above**. The cell shows the trace (model, tokens, time) and then checks every number in the card against the fact sheet: that is the hallucination check. On the Lexsi platform this is the job of the Attest agent.

In [ ]:
# 📝 Model card written by an open-source LLM from the recorded facts, with a full trace and a grounding check
LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"        # open weights (Apache-2.0), runs on the A100; set None to use the template writer
USE_LLM = DEVICE == "cuda" and not FAST and LLM_MODEL is not None
_LLM = {}
CARD_SECTIONS = ["1. Model overview and intended use", "2. Data and preparation", "3. Model selection and training",
                 "4. Performance on held-out data", "5. Explainability", "6. Uncertainty, fairness and limitations",
                 "7. Monitoring and retraining plan", "8. Governance and regulatory alignment", "9. Change log"]
SYSTEM_PROMPT = ("You are a model-risk documentation writer for a bank regulated by SAMA and the CBB. Write a model card in Markdown with "
                 "exactly these sections: " + "; ".join(CARD_SECTIONS) + ". Use ONLY the facts in the JSON the user gives you. "
                 "Quote numbers exactly as given. If a fact is missing, write 'not recorded'. Do not invent figures, dates, clause numbers or approvals. "
                 "In section 8 name the frameworks listed in the facts and state that control mapping is owned by the bank's Compliance function.")


def _json_safe(o):
    if isinstance(o, dict):
        return {str(k): _json_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        return [_json_safe(v) for v in o]
    if isinstance(o, (np.floating, np.integer)):
        return float(o)
    if isinstance(o, (pd.Timestamp,)):
        return str(o)
    return o


def _template_card(facts, changes):
    """Deterministic writer used when no LLM is available: every line comes straight from the facts."""
    lines = [f"# Model card · {facts.get('use_case', {}).get('name', 'model')}", "", "*Written by the template writer (no LLM was run).*", ""]
    keys = list(facts)
    alloc = {0: ["use_case"], 1: ["data", "preparation", "features"], 2: ["selection", "training"], 3: ["performance"],
             4: ["explainability"], 5: ["uncertainty", "fairness", "limitations"], 6: ["monitoring", "retraining"], 7: ["governance"]}
    for i, title in enumerate(CARD_SECTIONS[:8]):
        lines += [f"## {title}"]
        rows = [(k, v) for sec in alloc[i] if sec in facts for k, v in facts[sec].items()]
        lines += [f"- **{k.replace('_', ' ')}:** {v}" for k, v in rows] or ["- not recorded"]
        lines.append("")
    lines += [f"## {CARD_SECTIONS[8]}"] + ([f"- {c}" for c in changes] if changes else ["- Initial version."])
    return "\n".join(lines)


def generate_model_card(facts, changes=None, version_label="v1"):
    facts = _json_safe(facts)
    user = "FACTS (JSON):\n" + json.dumps(facts, indent=1) + ("\n\nCHANGES SINCE THE PREVIOUS VERSION:\n- " + "\n- ".join(changes) if changes else "")
    trace = {"version": version_label, "writer": LLM_MODEL if USE_LLM else "template writer", "prompt_characters": len(SYSTEM_PROMPT) + len(user)}
    t0 = time.time()
    if USE_LLM:
        try:
            from transformers import AutoModelForCausalLM, AutoTokenizer
            if "model" not in _LLM:
                _LLM["tok"] = AutoTokenizer.from_pretrained(LLM_MODEL)
                _LLM["model"] = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=torch.bfloat16, device_map="auto")
            tok, model = _LLM["tok"], _LLM["model"]
            ids = tok.apply_chat_template([{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}],
                                          add_generation_prompt=True, return_tensors="pt").to(model.device)
            out = model.generate(ids, max_new_tokens=1400, do_sample=False)
            card = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
            trace.update(input_tokens=int(ids.shape[1]), output_tokens=int(out.shape[1] - ids.shape[1]))
        except Exception as exc:
            trace["writer"] = f"template writer (LLM failed: {str(exc)[:80]})"
            card = _template_card(facts, changes)
    else:
        card = _template_card(facts, changes)
    trace["seconds"] = round(time.time() - t0, 1)

    # Grounding check: every number in the card must appear in the facts (or the change list)
    source = json.dumps(facts) + " " + " ".join(changes or [])
    known = {n.rstrip("0").rstrip(".") if "." in n else n for n in re.findall(r"\d+(?:\.\d+)?", source.replace(",", ""))}
    found = re.findall(r"\d+(?:\.\d+)?", re.sub(r"^#+ \d+\.", "", card.replace(",", ""), flags=re.M))
    norm = lambda n: n.rstrip("0").rstrip(".") if "." in n else n
    checked = [n for n in found if not (n.isdigit() and int(n) < 13)]              # skip list numbering and small counts
    unverified = sorted({n for n in checked if norm(n) not in known})
    missing = [s for s in CARD_SECTIONS if s.split(". ", 1)[1].split(" ")[0].lower() not in card.lower()]
    trace.update(numbers_checked=len(checked), numbers_not_in_facts=len(unverified),
                 grounded_pct=round(100 * (1 - len(unverified) / max(len(checked), 1)), 1), sections_missing=len(missing))
    return card, trace, unverified


def show_card(card, trace, unverified):
    display(pd.DataFrame([trace]).T.rename(columns={0: "trace"}))
    fig = go.Figure(go.Indicator(mode="gauge+number", value=trace["grounded_pct"], number={"suffix": "%"},
                                 title={"text": "Numbers in the card that trace back to recorded facts"},
                                 gauge={"axis": {"range": [0, 100]}, "bar": {"color": TEAL if trace["grounded_pct"] >= 99 else RED}}))
    fig.update_layout(height=260, margin=dict(t=60, b=10)); fig.show()
    if unverified:
        print("numbers NOT found in the facts (review before use):", unverified[:20])
    display(Markdown(card))

In [ ]:
# 📝 Generate model card v1
record("use_case", name=USE_CASE_NAME, task="probabilistic time-series forecasting with anomaly monitoring", decision_supported=DECISION_SUPPORTED, owner="[model owner]", version="v1")
record("monitoring", plan="daily anomaly scoring of new observations; window-by-window error, coverage and drift checks; champion-challenger review by an agent with human approval")
CARD_V1, TRACE_V1, UNVERIFIED_V1 = generate_model_card(FACTS, version_label="v1")
show_card(CARD_V1, TRACE_V1, UNVERIFIED_V1)
with open("model_card_v1.md", "w") as f:
    f.write(CARD_V1)

## 8.2 Production API contract
The envelope follows the Lexsi case call (project, unique identifier, tag, data); the payload is a forecasting request: recent history, the calendar for the future window, the horizon and the quantiles wanted. The cell sends a real request and answers it with the selected model.

In [ ]:
# 🔌 Forecast API: contract, validation and a live round trip
def forecast_api(request):
    t0 = time.time()
    for field in ("unique_identifier", "horizon", "data"):
        if field not in request:
            return {"success": False, "details": f"missing field: {field}"}
    item = request["unique_identifier"]
    if item not in SCALE.index:
        return {"success": False, "details": f"unknown series: {item}"}
    hist_ = pd.DataFrame(request["data"]["history"]); hist_["timestamp"] = pd.to_datetime(hist_["timestamp"])
    hist_ = hist_.assign(item_id=item, y=hist_["value"] / SCALE[item])[["item_id", "timestamp", "y"]]
    p = TimeSeriesPipeline(PRIMARY, forecast_params={"prediction_length": int(request["horizon"]), "quantile_levels": request.get("quantile_levels", QUANTILES)},
                           tuning_params={"device": DEVICE}, model_params=dict(PRIMARY_PARAMS))
    out = p.fit(hist_, schema).predict().to_pandas()
    qcols = [c for c in out.columns if c not in ("item_id", "timestamp", "target")]
    out[qcols] = out[qcols] * SCALE[item]
    return {"success": True, "model_name": PRIMARY, "model_version": "v1", "unique_identifier": item, "unit": UNIT,
            "forecast": [{"timestamp": str(r.timestamp.date()), **{c: round(float(out.loc[r.Index, c]), 2) for c in qcols}}
                         for r in out.itertuples()],
            "audit_trail": {"latency_ms": round((time.time() - t0) * 1000), "context_points": len(hist_), "scored_at": pd.Timestamp.now().isoformat(timespec="seconds")}}

api_item = meta.item_id.iloc[0]
h_ = value_hist[value_hist.item_id == api_item].tail(max(SCALE_WINDOW, 8 * MONTH_LAG))
request = {"project_name": "[lexsi project]", "unique_identifier": api_item, "tag": "Live", "horizon": HORIZON, "quantile_levels": QUANTILES,
           "data": {"history": [{"timestamp": str(t.date()), "value": round(float(v), 4)} for t, v in zip(h_.timestamp, h_.value)]}}
response = forecast_api(request)
print("POST /forecast   headers: x-api-token\n\nREQUEST (history truncated to 3 points)")
print(json.dumps({**request, "data": {"history": request["data"]["history"][:3] + ["…"]}}, indent=1))
print("\nRESPONSE (first 3 steps)")
print(json.dumps({**response, "forecast": response.get("forecast", [])[:3]}, indent=1))
print("\nrejects a bad request:", forecast_api({"unique_identifier": "no such series", "horizon": 5, "data": {}})["details"])
if response["success"]:
    r_ = pd.DataFrame(response["forecast"]); r_["timestamp"] = pd.to_datetime(r_["timestamp"])
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=h_.timestamp.tail(120), y=h_.value.tail(120), name="history sent", line=dict(color=DARK)))
    fig.add_trace(go.Scatter(x=pd.concat([r_.timestamp, r_.timestamp[::-1]]), y=pd.concat([r_["0.95"], r_["0.05"][::-1]]), fill="toself", fillcolor="rgba(47,91,234,0.15)", line=dict(width=0), name="90% band returned"))
    fig.add_trace(go.Scatter(x=r_.timestamp, y=r_["0.5"], name="median returned", line=dict(color=BLUE, width=3)))
    fig.update_layout(template="plotly_white", height=380, title=f"API round trip · {api_item}", yaxis_title=UNIT)
    fig.show()

# 9 · Model in production

## 9.1 Anomaly replay: the COVID fiscal shock

Real data has no anomaly labels, so we replay history. The detector watches January 2019 to June 2021, with 2019 as its clean reference period, and each day is scored only from earlier data. A useful monitor should stay quiet through 2019 and light up from March 2020, when stimulus payments, unemployment benefits and deferred tax deadlines broke every seasonal pattern.

Because there are no labels, this section reports **where and when alerts fire**, not a detection score.

In [ ]:
def monitor_figure(res, feed_df, items, title_suffix, truth=None, events=None):
    """Expected band, observed flow, alerts (and true incidents or known events) with a series dropdown."""
    fig = go.Figure()
    per = 5
    for k, item in enumerate(items):
        r = res[res.item_id == item]; f = feed_df[feed_df.item_id == item]; vis = k == 0
        fig.add_trace(go.Scatter(x=pd.concat([r.timestamp, r.timestamp[::-1]]), y=pd.concat([r.upper, r.lower[::-1]]),
                                 fill="toself", fillcolor="rgba(47,91,234,0.14)", line=dict(width=0), name="expected range", visible=vis))
        fig.add_trace(go.Scatter(x=f.timestamp, y=f.y, name="observed", line=dict(color=DARK, width=1.3), visible=vis))
        if truth is not None:
            t = r.merge(truth[truth.is_anomaly], on=["item_id", "timestamp"])
            fig.add_trace(go.Scatter(x=t.timestamp, y=t.value, mode="markers", name="true incident", visible=vis,
                                     marker=dict(symbol="circle-open", size=14, color=ORANGE, line=dict(width=2))))
        else:
            fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name="", showlegend=False, visible=vis))
        a = r[r.is_anomaly.fillna(False).astype(bool)]
        fig.add_trace(go.Scatter(x=a.timestamp, y=a.value, mode="markers", name="TabTune alert", visible=vis,
                                 marker=dict(symbol="x", size=9, color=RED)))
        fig.add_trace(go.Scatter(x=r.timestamp, y=r.score, name="anomaly score", yaxis="y2", line=dict(color=GREY, width=1), visible=vis))
    buttons = [dict(label=item, method="update",
                    args=[{"visible": [j // per == k for j in range(per * len(items))]}, {"title": f"{item} · {title_suffix}"}])
               for k, item in enumerate(items)]
    for when, label in (events or []):
        fig.add_vline(x=pd.Timestamp(when), line_dash="dot", line_color=GOLD)
        fig.add_annotation(x=pd.Timestamp(when), y=1.02, yref="paper", text=label, showarrow=False, textangle=-20, font=dict(size=10))
    fig.update_layout(template="plotly_white", height=580, title=f"{items[0]} · {title_suffix}",
                      yaxis=dict(title="indexed flow", domain=[0.3, 1]), yaxis2=dict(title="score", domain=[0, 0.22]),
                      updatemenus=[dict(buttons=buttons, x=1.0, xanchor="right", y=1.18)], legend=dict(orientation="h", y=-0.12))
    return fig

In [ ]:
# 🚨 Replay January 2019 – June 2021
EVENTS = [("2020-03-27", "CARES Act"), ("2020-04-15", "stimulus payments"), ("2020-07-15", "deferred tax deadline"),
          ("2021-01-04", "second stimulus"), ("2021-03-17", "third stimulus")]
replay = panel[(panel.timestamp >= "2019-01-01") & (panel.timestamp <= "2021-06-30")][["item_id", "timestamp", "y"]].reset_index(drop=True)
t0 = time.time()
det = TimeSeriesPipeline(DETECTOR, task_type="anomaly_detection", tuning_params={"device": DEVICE},
                         task_params={"method": "forecast_error", "min_context": 120, "alpha": 0.01, "reference_fraction": 0.25})
res = det.fit(replay, schema).predict().to_pandas()
print(f"scored {int(res.score.notna().sum()):,} series-days in {time.time() - t0:.0f}s · alerts: {int(res.is_anomaly.sum())}")

watch = ["Unemployment benefits", "Individual tax refunds", "Individual taxes, not withheld", "Other withdrawals", "Medicaid", "Federal tax deposits"]
monitor_figure(res, replay, watch + [i for i in meta.item_id if i not in watch], "anomaly replay", events=EVENTS).show()

In [ ]:
# 📋 Where and when the alerts fired
alerts = res[res.is_anomaly.fillna(False).astype(bool)].copy()
scored = res[res.score.notna()]
monthly = (alerts.groupby(["item_id", pd.Grouper(key="timestamp", freq="MS")]).size().unstack(fill_value=0)
           .reindex(index=meta.item_id, fill_value=0))
months = pd.date_range(scored.timestamp.min().to_period("M").to_timestamp(), scored.timestamp.max(), freq="MS")
monthly = monthly.reindex(columns=months, fill_value=0)
rate = (alerts.groupby(pd.Grouper(key="timestamp", freq="MS")).size().reindex(months, fill_value=0)
        / scored.groupby(pd.Grouper(key="timestamp", freq="MS")).size().reindex(months).clip(lower=1) * 100)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3], vertical_spacing=0.05,
                    subplot_titles=("Alerts per line per month", "Share of series-days flagged (%)"))
fig.add_trace(go.Heatmap(z=monthly.values, x=monthly.columns, y=monthly.index, colorscale=[[0, "#F7F7F4"], [1, RED]], zmin=0,
                         colorbar=dict(title="alerts", len=0.6, y=0.7)), 1, 1)
fig.add_trace(go.Bar(x=rate.index, y=rate.values, marker_color=[RED if d >= pd.Timestamp("2020-03-01") else GREY for d in rate.index],
                     showlegend=False), 2, 1)
fig.add_hline(y=1.0, line_dash="dot", line_color=DARK, row=2, col=1)
for when, label in EVENTS:
    fig.add_vline(x=pd.Timestamp(when), line_dash="dot", line_color=GOLD)
fig.update_layout(template="plotly_white", height=760, title=f"Anomaly replay · {DETECTOR} · dotted line = the 1% false-alarm rate the detector was set to")
fig.show()

pre = scored.timestamp < "2020-03-01"
summary = pd.DataFrame({"series-days scored": [int(pre.sum()), int((~pre).sum())],
                        "alerts": [int((alerts.timestamp < "2020-03-01").sum()), int((alerts.timestamp >= "2020-03-01").sum())]},
                       index=["before March 2020", "March 2020 onwards"])
summary["alert rate %"] = summary.alerts / summary["series-days scored"] * 100
display(summary.style.format({"alert rate %": "{:.2f}%"}))
top = alerts.assign(deviation=(alerts.value - alerts.expected)).sort_values("score", ascending=False).head(12)
display(top[["timestamp", "item_id", "value", "expected", "score", "p_value"]].style.format({"value": "{:.2f}", "expected": "{:.2f}", "score": "{:.1f}", "p_value": "{:.4f}"})
        .set_caption("Strongest alerts (indexed units: 1.0 = the line's average day)"))

## 9.2 Drift monitoring and the retraining agent
The last three forecast windows are replayed in order. In each one the agent **observes** error, band coverage and level drift (is this window's average unusual compared with past windows of the same length?) for the model in production. At the scheduled review it **proposes challengers**, asks a **human to approve** the review, **verifies** champion and challengers on the next unseen window, and **promotes** a challenger only if it wins by a margin and a human signs off.

For a zero-shot forecaster "retraining" means refreshing what can change: the model choice, the ensemble weights, the interval calibration and, optionally, LoRA fine-tuning (`RUN_FINETUNE = True`).

In [ ]:
# 📡 Replay three windows: error, coverage and drift for the model in production
def level_shift(ref_y, new_y):
    """How unusual is this window's average level? z-score against same-length windows in the reference period."""
    roll = pd.Series(ref_y).rolling(len(new_y)).mean().dropna()
    return float((np.mean(new_y) - roll.mean()) / (roll.std() + 1e-9)) if len(roll) > 5 else 0.0

def run_champion(hist_df, act_df):
    p = TimeSeriesPipeline(PRIMARY, forecast_params=fp, license_mode="commercial", tuning_params={"device": DEVICE}, model_params=dict(PRIMARY_PARAMS))
    p.fit(hist_df, schema_cov, future_df=act_df[["item_id", "timestamp", *KNOWN]])
    p.calibrate(windows=1 if FAST else 3)
    return p.evaluate(act_df, forecast=p.predict())

h_w2, a_w2 = split_horizon(history, schema_cov, prediction_length=HORIZON)
h_w1, a_w1 = split_horizon(h_w2, schema_cov, prediction_length=HORIZON)
WINDOWS = [("W1", h_w1, a_w1), ("W2", h_w2, a_w2), ("W3", history, actual)]
COV_KEY = COVK[-1]; cov_scale = 100 if m_cal[COV_KEY] <= 1 else 1
mon = []
for name, h_df, a_df in WINDOWS:
    mm = m_cal if name == "W3" else run_champion(h_df, a_df)
    ref = h_df.groupby("item_id").tail(2 * SCALE_WINDOW)
    drift = pd.Series({i: level_shift(ref[ref.item_id == i].y.to_numpy(), g.y.to_numpy()) for i, g in a_df.groupby("item_id")})
    mon.append({"window": name, "start": a_df.timestamp.min().date(), "mase": mm["mase"], "wql": mm["wql"], "coverage": mm[COV_KEY] * cov_scale,
                "drifted_series": int((drift.abs() > 2.5).sum()), "psi": drift})
mon_tab = pd.DataFrame([{k: v for k, v in m.items() if k != "psi"} for m in mon])
psi_grid = pd.DataFrame({m["window"]: m["psi"] for m in mon}).reindex(meta.sort_values("group").item_id)
fig = make_subplots(rows=1, cols=3, column_widths=[0.27, 0.27, 0.46], horizontal_spacing=0.17,
                    subplot_titles=("Forecast error by window (MASE)", f"Coverage of the {COV_KEY.split('_')[1]}% band (%)", "Level drift by series (z-score vs past windows; beyond ±2.5 = material)"))
fig.add_trace(go.Scatter(x=mon_tab.window, y=mon_tab.mase, mode="lines+markers", line=dict(color=BLUE, width=3), showlegend=False), 1, 1)
fig.add_hline(y=FACTS["selection"]["selected_backtest_mase"], line_dash="dot", line_color=GREY, row=1, col=1)
fig.add_trace(go.Bar(x=mon_tab.window, y=mon_tab.coverage, marker_color=TEAL, showlegend=False, text=mon_tab.coverage.round(0), textposition="outside"), 1, 2)
fig.add_hline(y=float(COV_KEY.split("_")[1]), line_dash="dot", line_color=DARK, row=1, col=2)
fig.add_trace(go.Heatmap(z=psi_grid.values, x=psi_grid.columns, y=psi_grid.index, colorscale="RdBu_r", zmin=-4, zmax=4, colorbar=dict(title="z")), 1, 3)
fig.update_layout(template="plotly_white", height=max(460, 18 * len(psi_grid)), title=f"Production monitoring · {PRIMARY} · three consecutive {HORIZON}-step windows")
fig.show()
display(mon_tab.style.hide(axis="index").format({"mase": "{:.3f}", "wql": "{:.3f}", "coverage": "{:.1f}"}))

In [ ]:
# 🔁 Retraining agent: observe → decide → approve → build challengers → verify → approve → promote
MASE_ALERT, COVERAGE_ALERT, DRIFT_ALERT, PROMOTE_MARGIN = 1.15, 10.0, 0.25, 0.01     # DRIFT_ALERT = share of series drifted
release_mase = FACTS["selection"]["selected_backtest_mase"]; promised_cov = float(COV_KEY.split("_")[1])
state = {"champion": PRIMARY, "version": 1, "events": []}
for i, m in enumerate(mon[:2]):
    reasons = []
    if m["mase"] > MASE_ALERT * release_mase:
        reasons.append(f"MASE {m['mase']:.3f} is more than {MASE_ALERT - 1:.0%} above the release level {release_mase:.3f}")
    if abs(m["coverage"] - promised_cov) > COVERAGE_ALERT:
        reasons.append(f"band coverage {m['coverage']:.0f}% vs {promised_cov:.0f}% promised")
    if m["drifted_series"] > DRIFT_ALERT * len(m["psi"]):
        reasons.append(f"{m['drifted_series']} series drifted")
    if i == 1:
        reasons.append("scheduled periodic review")
    audit("retrain-agent", f"{m['window']} observe", f"MASE {m['mase']:.3f}, coverage {m['coverage']:.0f}%, {m['drifted_series']} drifted series",
          "REVIEW PROPOSED" if (reasons and i == 1) else ("alert raised to the model owner" if reasons else "no action"))
    if i == 1 and reasons and human_gate(f"{m['window']}: run a champion-challenger review ({'; '.join(reasons)})?", True):
        challengers = {}
        def try_challenger(label, build):
            try:
                t0 = time.time(); res_ = build(); challengers[label] = {"mase": res_["mase"], "wql": res_["wql"], "seconds": time.time() - t0}
            except Exception as exc:
                audit("retrain-agent", "challenger failed", f"{label}: {str(exc)[:90]}")
        if LB_WINNER != PRIMARY:
            def _winner():
                p = TimeSeriesPipeline(LB_WINNER, forecast_params=fp, tuning_params={"device": DEVICE},
                                       model_params=dict(GBM_PARAMS if LB_WINNER == "TabularTS-GBM" else {})).fit(history_y, schema)
                p.calibrate(windows=1 if FAST else 3); return p.evaluate(actual, forecast=p.predict())
            try_challenger(f"{LB_WINNER} (recalibrated)", _winner)
        try_challenger("Greedy ensemble (refreshed weights)", lambda: m_ens)
        if RUN_FINETUNE and not FAST:
            try_challenger("ChronosBolt + LoRA (retrained)", lambda: TimeSeriesPipeline("ChronosBolt", tuning_strategy="peft", forecast_params=fp,
                           tuning_params={"epochs": 2, "steps_per_epoch": 50, "device": DEVICE}).fit(history_y, schema).evaluate(actual))
        champ = {"mase": m_cal["mase"], "wql": m_cal["wql"]}
        best = min(challengers, key=lambda k: challengers[k]["wql"]) if challengers else None
        win = bool(best) and challengers[best]["wql"] < champ["wql"] * (1 - PROMOTE_MARGIN) and challengers[best]["mase"] <= champ["mase"] * 1.02
        audit("retrain-agent", "W3 verify", f"champion {PRIMARY}: MASE {champ['mase']:.3f}, WQL {champ['wql']:.3f}; best challenger {best}: "
              + (f"MASE {challengers[best]['mase']:.3f}, WQL {challengers[best]['wql']:.3f}" if best else "none"))
        promoted = win and human_gate(f"Promote '{best}' (WQL {challengers[best]['wql']:.3f} vs {champ['wql']:.3f})?", True)
        if promoted:
            state.update(champion=best, version=2)
        audit("retrain-agent", "promote", f"needs WQL better by {PROMOTE_MARGIN:.0%} with MASE no worse than +2%", f"PROMOTED {best} as v2" if promoted else f"KEPT {PRIMARY}")
        state["events"].append({"window": m["window"], "reasons": reasons, "champion": champ, "challengers": challengers, "best": best, "promoted": promoted})

if state["events"]:
    e = state["events"][0]
    names = [f"{PRIMARY} (champion)"] + list(e["challengers"]); vals = [e["champion"]] + list(e["challengers"].values())
    fig = make_subplots(rows=1, cols=2, subplot_titles=("MASE on the next unseen window (W3)", "WQL on the next unseen window (W3)"))
    for c_, key in enumerate(("mase", "wql"), 1):
        fig.add_trace(go.Bar(x=names, y=[v[key] for v in vals], showlegend=False, text=[f"{v[key]:.3f}" for v in vals], textposition="outside",
                             marker_color=[BLUE if (n.startswith(state["champion"])) else GREY for n in names]), 1, c_)
    fig.update_layout(template="plotly_white", height=440, title=f"Champion vs challengers · in production now: {state['champion']} (v{state['version']})")
    fig.show()
else:
    print("No review was run.")

## 9.3 Regenerated model card
The card is rewritten from the updated fact sheet, with a change log. The same grounding check runs again.

In [ ]:
# 📝 Model card v2 after the production period
changes = []
record("monitoring", windows_replayed=len(mon), worst_window_mase=float(mon_tab.mase.max()), lowest_window_coverage_pct=float(mon_tab.coverage.min()),
       max_drifted_series_in_a_window=int(mon_tab.drifted_series.max()), anomaly_detector=DETECTOR)
if state["events"]:
    e = state["events"][0]
    record("retraining", review_window=e["window"], trigger="; ".join(e["reasons"]), challengers_tested=len(e["challengers"]), champion_wql_next_window=e["champion"]["wql"],
           best_challenger=str(e["best"]), best_challenger_wql_next_window=float(e["challengers"][e["best"]]["wql"]) if e["best"] else float("nan"),
           outcome=f"{e['best']} promoted" if e["promoted"] else f"{PRIMARY} kept", human_approvals="review and promotion gates")
    changes = [f"Review run after {e['window']}: {'; '.join(e['reasons'])}.",
               f"{len(e['challengers'])} challenger(s) verified on the next unseen window against the champion (WQL {e['champion']['wql']:.4f}).",
               (f"{e['best']} promoted after human approval." if e["promoted"] else f"{PRIMARY} kept: no challenger cleared the promotion margin.")]
else:
    record("retraining", outcome="no review was run during the monitoring period")
    changes = ["Monitoring period completed with no review."]
FACTS["use_case"]["version"] = f"v{state['version']}" if state["version"] > 1 else "v1 (reviewed)"
CARD_V2, TRACE_V2, UNVERIFIED_V2 = generate_model_card(FACTS, changes=changes, version_label="v2")
show_card(CARD_V2, TRACE_V2, UNVERIFIED_V2)
with open("model_card_v2.md", "w") as f:
    f.write(CARD_V2)
import difflib
d_ = list(difflib.unified_diff(CARD_V1.splitlines(), CARD_V2.splitlines(), lineterm="", n=0))
print(f"\nmodel card diff v1 → v2: {sum(l.startswith('+') and not l.startswith('+++') for l in d_)} lines added, "
      f"{sum(l.startswith('-') and not l.startswith('---') for l in d_)} removed")

# 10 · Conclusions

In [ ]:
# 🧾 Build the plain-English conclusions from the recorded facts
P, S, U = FACTS["performance"], FACTS["selection"], FACTS["uncertainty"]
CONCLUSIONS = [
    f"**Data:** {FACTS['data']['series']:.0f} series and {FACTS['data']['panel_rows']:,.0f} rows were prepared and checked by a data readiness agent, with human approval at each step.",
    f"**Model:** {S['candidates_succeeded']:.0f} forecasters were raced on rolling backtests; **{S['selected_model']}** was selected (backtest MASE {S['selected_backtest_mase']:.3f}; seasonal-naive is {S['seasonal_naive_mase']:.3f}).",
    f"**Accuracy on unseen data:** MASE {P['mase']:.3f} (below 1 beats the seasonal-naive forecast) and a typical error of {P['wape_pct']:.0f}% of the flow.",
    f"**Uncertainty:** the {U['promised_coverage_pct']:.0f}% band held {U['measured_coverage_pct']:.0f}% of actual outcomes.",
    f"**Drivers:** the covariates the model leaned on most were {FACTS['explainability']['most_useful_covariates']}.",
    f"**Documentation:** the model card was written by {TRACE_V2['writer']}; {TRACE_V2['grounded_pct']:.0f}% of its numbers trace back to recorded facts.",
    f"**Production:** three windows were replayed with drift, coverage and error checks; review outcome: {FACTS['retraining']['outcome']}.",
    CLOSING_LINE,
]

In [ ]:
# ✅ Conclusions in plain English, and the full audit trail
display(Markdown("## What this exercise showed\n\n" + "\n".join(f"- {line}" for line in CONCLUSIONS)))
trail = pd.DataFrame(AUDIT)
if len(trail):
    fig = px.histogram(trail, x="step", color="actor", color_discrete_sequence=PALETTE, title="Audit trail · logged agent steps and human decisions")
    fig.update_layout(template="plotly_white", height=360, xaxis_title=""); fig.show()
    display(trail.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))